[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/linear_algebra/09_numerical_spectrum_algorithms/exercises.ipynb)

# Module 09 — Exercises: Numerical Spectrum Algorithms

Forty-three solved problems in four tiers. Every problem carries a statement, a one-line
intuition, a stepwise solution, a boxed answer, a key takeaway, and — wherever the answer is
numeric or algorithmic — a code cell that recomputes it.

Theorem, proof, definition and example numbers refer to
[first_principles.ipynb](first_principles.ipynb). Symbols follow
[the notation register](../../docs/notation.md): eigenvalues in descending modulus, norms written
$\lVert x \rVert$, transposes written $A^{\top}$, transition matrices column-stochastic.

The preamble below is shared by every code cell in this notebook.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.0, 4.0),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

EPS = np.finfo(float).eps
print(f"machine epsilon = {EPS:.4e}")


def qr_positive(M):
    """QR factorization normalized so that R has a positive diagonal."""
    Q, R = np.linalg.qr(M)
    sgn = np.sign(np.diag(R))
    sgn[sgn == 0.0] = 1.0
    return Q * sgn, sgn[:, None] * R


def wilkinson_shift(a, b, c):
    """Eigenvalue of [[a, b], [b, c]] nearest c, in the cancellation-free form."""
    d = (a - c) / 2.0
    sgn = 1.0 if d >= 0.0 else -1.0
    return c - b * b / (d + sgn * np.sqrt(d * d + b * b))


def lanczos(A, b, m, reorth=True):
    """m steps of Lanczos; returns alphas, betas and the basis Q (n x m+1)."""
    nn = A.shape[0]
    Qb = np.zeros((nn, m + 1))
    a = np.zeros(m)
    bt = np.zeros(m)
    Qb[:, 0] = b / np.linalg.norm(b)
    for j in range(m):
        w = A @ Qb[:, j]
        a[j] = Qb[:, j] @ w
        w = w - a[j] * Qb[:, j] - (bt[j - 1] * Qb[:, j - 1] if j > 0 else 0.0)
        if reorth:
            w = w - Qb[:, :j + 1] @ (Qb[:, :j + 1].T @ w)
            w = w - Qb[:, :j + 1] @ (Qb[:, :j + 1].T @ w)
        bt[j] = np.linalg.norm(w)
        Qb[:, j + 1] = w / bt[j] if bt[j] > 0.0 else 0.0
    return a, bt, Qb

machine epsilon = 2.2204e-16


## L0 — Concept Checks

### Problem L0.1 — Rate of power iteration from a spectrum

**Statement.** A diagonalizable $A \in \mathbb{R}^{3 \times 3}$ has eigenvalues $5$, $-2$, $1$.
Give the asymptotic per-step contraction factor of the eigenvector error in power iteration.

**Intuition.** Only the two largest **moduli** matter; the sign of $\lambda_2$ is irrelevant.

**Solution.**

*Step 1.* Order by modulus: $\lvert \lambda_1 \rvert = 5$, $\lvert \lambda_2 \rvert = 2$,
$\lvert \lambda_3 \rvert = 1$.

*Step 2.* By the power-iteration rate of
[Module 06](../06_eigenvalues_eigenvectors_spectral_theory/first_principles.ipynb), Theorem 4.7,
the factor is $\lvert \lambda_2/\lambda_1 \rvert$.

$$
\boxed{\left\lvert \frac{-2}{5} \right\rvert = \frac{2}{5} = 0.4}
$$

**Key takeaway.** Power iteration is linearly convergent at the modulus ratio of the two
dominant eigenvalues; a negative $\lambda_2$ changes the sign pattern of the iterates, never the
speed.

In [2]:
lam = np.array([5.0, -2.0, 1.0])
Q3, _ = np.linalg.qr(rng.standard_normal((3, 3)))
A = Q3 @ np.diag(lam) @ Q3.T
x = rng.standard_normal(3)
x /= np.linalg.norm(x)
q1 = Q3[:, 0]
sins = []
for _ in range(30):
    sins.append(np.linalg.norm(x - (q1 @ x) * q1))
    x = A @ x
    x /= np.linalg.norm(x)
sins = np.array(sins)
obs = np.exp(np.polyfit(np.arange(10, 25), np.log(sins[10:25]), 1)[0])
print(f"observed contraction {obs:.6f}   predicted |lambda_2/lambda_1| = {abs(lam[1]/lam[0]):.6f}")
assert abs(obs - 0.4) < 1e-4

observed contraction 0.400000   predicted |lambda_2/lambda_1| = 0.400000


### Problem L0.2 — Which eigenvalue does a shift select?

**Statement.** $\operatorname{spec}(A) = \lbrace 10, 4, 1, -3 \rbrace$. Shifted inverse iteration
is run with $\mu = 3.5$. Which eigenvalue does it find, and at what rate?

**Intuition.** Inverting sends $\lambda_i \mapsto (\lambda_i - \mu)^{-1}$, so the eigenvalue
closest to $\mu$ becomes the dominant one.

**Solution.**

*Step 1.* Distances to $\mu = 3.5$: $6.5$, $0.5$, $2.5$, $6.5$.

*Step 2.* The closest is $\lambda = 4$ at distance $0.5$; the second closest is $\lambda = 1$ at
distance $2.5$.

*Step 3.* Theorem 4.1 gives the rate as the ratio of those two distances.

$$
\boxed{\lambda = 4, \qquad r = \frac{0.5}{2.5} = \frac{1}{5} = 0.2}
$$

**Key takeaway.** A shift buys speed: moving $\mu$ from $3.5$ to $3.9$ would drop $r$ to
$0.1/2.9 \approx 0.034$.

In [3]:
spec = np.array([10.0, 4.0, 1.0, -3.0])
mu = 3.5
Q4, _ = np.linalg.qr(rng.standard_normal((4, 4)))
A = Q4 @ np.diag(spec) @ Q4.T
A = (A + A.T) / 2.0
dist = np.abs(spec - mu)
j = int(np.argmin(dist))
rate = dist[j] / np.min(np.delete(dist, j))
x = rng.standard_normal(4)
x /= np.linalg.norm(x)
qj = Q4[:, j]
sins = []
for _ in range(20):
    sins.append(np.linalg.norm(x - (qj @ x) * qj))
    y = np.linalg.solve(A - mu * np.eye(4), x)
    x = y / np.linalg.norm(y)
obs = np.exp(np.polyfit(np.arange(5, 15), np.log(np.array(sins)[5:15]), 1)[0])
print(f"target eigenvalue {spec[j]}   Rayleigh quotient {x @ A @ x:.10f}")
print(f"observed rate {obs:.6f}   predicted {rate:.6f}")
assert abs(spec[j] - 4.0) < 1e-12 and abs(obs - 0.2) < 1e-4

target eigenvalue 4.0   Rayleigh quotient 4.0000000000
observed rate 0.200000   predicted 0.200000


### Problem L0.3 — A Rayleigh quotient

**Statement.** For $A = \left(\begin{smallmatrix}3 & 1 \\ 1 & 3\end{smallmatrix}\right)$ and
$u = \tfrac{1}{\sqrt5}(2, 1)^{\top}$, compute $R_A(u)$.

**Intuition.** $u$ is a unit vector, so $R_A(u) = u^{\top}Au$ with no denominator to worry about.

**Solution.**

*Step 1.* $Au = \tfrac{1}{\sqrt5}(3 \cdot 2 + 1, \ 2 + 3 \cdot 1)^{\top} = \tfrac{1}{\sqrt5}(7, 5)^{\top}$.

*Step 2.* $u^{\top}(Au) = \tfrac{1}{5}(2 \cdot 7 + 1 \cdot 5) = \tfrac{19}{5}$.

$$
\boxed{R_A(u) = \frac{19}{5} = 3.8}
$$

**Key takeaway.** The value lies strictly between the eigenvalues $2$ and $4$, as every Rayleigh
quotient must: it is a weighted average of eigenvalues.

In [4]:
A = np.array([[3.0, 1.0], [1.0, 3.0]])
u = np.array([2.0, 1.0]) / np.sqrt(5.0)
print("R_A(u) =", u @ A @ u, "  19/5 =", 19 / 5)
print("eigenvalues:", np.linalg.eigvalsh(A))
assert abs(u @ A @ u - 19 / 5) < 1e-12

R_A(u) = 3.8   19/5 = 3.8
eigenvalues: [2. 4.]


### Problem L0.4 — Gershgorin discs

**Statement.** Give the Gershgorin discs of
$A = \left(\begin{smallmatrix}4 & 1 & 0 \\ 1 & 0 & 1 \\ 0 & 2 & 8\end{smallmatrix}\right)$ and say
how many eigenvalues each contains.

**Intuition.** Each row gives a disc centred at its diagonal entry with radius the sum of the
other entries in that row.

**Solution.**

*Step 1.* Centres $4, 0, 8$; radii $1, 2, 2$.

*Step 2.* The discs are $D_1 = \lbrace \lvert z-4 \rvert \le 1 \rbrace$,
$D_2 = \lbrace \lvert z \rvert \le 2 \rbrace$, $D_3 = \lbrace \lvert z-8 \rvert \le 2 \rbrace$.

*Step 3.* On the real line they are $[3,5]$, $[-2,2]$, $[6,10]$ — pairwise **disjoint**, so by the
Gershgorin refinement each holds exactly one eigenvalue.

$$
\boxed{[-2,2] \cup [3,5] \cup [6,10], \text{ one eigenvalue in each}}
$$

**Key takeaway.** Disjointness is worth more than containment: it converts a bound into a count,
which is exactly what a shift strategy needs.

In [5]:
A = np.array([[4.0, 1.0, 0.0], [1.0, 0.0, 1.0], [0.0, 2.0, 8.0]])
centres = np.diag(A)
radii = np.abs(A).sum(axis=1) - np.abs(centres)
ev = np.linalg.eigvals(A)
print("centres:", centres, " radii:", radii)
print("eigenvalues:", np.sort(ev.real))
for i in range(3):
    hits = [z for z in ev if abs(z - centres[i]) <= radii[i] + 1e-12]
    print(f"  disc {i+1} (centre {centres[i]}, radius {radii[i]}) contains {len(hits)} eigenvalue(s)")
    assert len(hits) == 1

centres: [4. 0. 8.]  radii: [1. 2. 2.]
eigenvalues: [-0.4606  4.211   8.2496]
  disc 1 (centre 4.0, radius 1.0) contains 1 eigenvalue(s)
  disc 2 (centre 0.0, radius 2.0) contains 1 eigenvalue(s)
  disc 3 (centre 8.0, radius 2.0) contains 1 eigenvalue(s)


### Problem L0.5 — A Wilkinson shift

**Statement.** Compute the Wilkinson shift for the trailing block
$B = \left(\begin{smallmatrix}3 & 1 \\ 1 & 5\end{smallmatrix}\right)$.

**Intuition.** It is the eigenvalue of $B$ nearer the corner entry $c = 5$.

**Solution.**

*Step 1.* $d = (3-5)/2 = -1$, so $\operatorname{sgn}(d) = -1$ and
$\sqrt{d^{2}+b^{2}} = \sqrt2$.

*Step 2.* Definition 3.5 gives

$$
\mu = 5 - \frac{1^{2}}{-1 - \sqrt2} = 5 + \frac{1}{1+\sqrt2} = 5 + (\sqrt2 - 1) .
$$

*Step 3.* The eigenvalues of $B$ are $4 \pm \sqrt2$; the nearer one to $5$ is $4 + \sqrt2$.

$$
\boxed{\mu = 4 + \sqrt2 \approx 5.41421}
$$

**Key takeaway.** The quotient form of Definition 3.5 is algebraically the same as
$c + d - \operatorname{sgn}(d)\sqrt{d^{2}+b^{2}}$ and numerically much better, because it never
subtracts two nearly equal numbers (Proposition 4.10).

In [6]:
mu = wilkinson_shift(3.0, 1.0, 5.0)
B = np.array([[3.0, 1.0], [1.0, 5.0]])
print(f"mu = {mu:.12f}   4 + sqrt2 = {4 + np.sqrt(2):.12f}")
print("eigenvalues of B:", np.linalg.eigvalsh(B))
print("distances from c = 5:", np.abs(np.linalg.eigvalsh(B) - 5.0))
assert abs(mu - (4 + np.sqrt(2))) < 1e-12

mu = 5.414213562373   4 + sqrt2 = 5.414213562373
eigenvalues of B: [2.5858 5.4142]
distances from c = 5: [2.4142 0.4142]


### Problem L0.6 — A Givens rotation

**Statement.** Build the Givens rotation $G$ that maps $(3, 4)^{\top}$ to $(r, 0)^{\top}$ with
$r \gt 0$, and give $r$.

**Intuition.** A plane rotation by the angle of the vector sends it onto the first axis.

**Solution.**

*Step 1.* $r = \sqrt{3^{2}+4^{2}} = 5$.

*Step 2.* $c = 3/5$, $s = 4/5$.

$$
\boxed{G = \begin{pmatrix} 3/5 & 4/5 \\ -4/5 & 3/5 \end{pmatrix}, \qquad r = 5}
$$

**Key takeaway.** One Givens rotation zeroes one entry and touches only two rows, so an
$n \times n$ Hessenberg QR step costs $O(n^{2})$ — the content of Theorem 4.5.

In [7]:
x = np.array([3.0, 4.0])
r = np.linalg.norm(x)
c, s = x[0] / r, x[1] / r
G = np.array([[c, s], [-s, c]])
print("G =\n", G, "\nG x =", G @ x, "  r =", r)
assert np.allclose(G @ x, [5.0, 0.0]) and np.allclose(G.T @ G, np.eye(2))

G =
 [[ 0.6  0.8]
 [-0.8  0.6]] 
G x = [ 5. -0.]   r = 5.0


### Problem L0.7 — Hotelling deflation

**Statement.** $A$ is symmetric with dominant eigenpair $(\lambda_1, v_1)$,
$\lVert v_1 \rVert = 1$. Write down a matrix $A'$ whose spectrum is that of $A$ with $\lambda_1$
replaced by $0$.

**Intuition.** Subtract the rank-one piece that $\lambda_1$ contributes to the spectral
decomposition.

**Solution.**

*Step 1.* Put $A' = A - \lambda_1 v_1 v_1^{\top}$.

*Step 2.* $A'v_1 = \lambda_1 v_1 - \lambda_1 v_1 = 0$.

*Step 3.* For any other eigenvector $v_i$, symmetry gives $v_1^{\top}v_i = 0$, so
$A'v_i = \lambda_i v_i$.

$$
\boxed{A' = A - \lambda_1 v_1 v_1^{\top}}
$$

**Key takeaway.** Deflation works cleanly for symmetric matrices because the eigenvectors are
orthogonal; for a non-symmetric matrix one must subtract using the **left** eigenvector, or the
other eigenvalues move.

In [8]:
M = rng.standard_normal((5, 5))
A = (M + M.T) / 2.0
lam, V = np.linalg.eigh(A)
i1 = int(np.argmax(np.abs(lam)))
Ad = A - lam[i1] * np.outer(V[:, i1], V[:, i1])
print("spectrum of A :", np.sort(lam))
print("spectrum of A':", np.sort(np.linalg.eigvalsh(Ad)))
expected = np.sort(np.concatenate([np.delete(lam, i1), [0.0]]))
assert np.allclose(np.sort(np.linalg.eigvalsh(Ad)), expected, atol=1e-12)

spectrum of A : [-1.248  -0.8688 -0.3183  1.6735  2.4381]
spectrum of A': [-1.248  -0.8688 -0.3183 -0.      1.6735]


### Problem L0.8 — The RQI shift at a given angle

**Statement.** For $A = \left(\begin{smallmatrix}2 & 1 \\ 1 & 2\end{smallmatrix}\right)$ and
$x_0 = (\cos\vartheta, \sin\vartheta)^{\top}$, give $\mu_0 = R_A(x_0)$ in closed form and evaluate
it at $\vartheta = 0.1$.

**Intuition.** Expand the quadratic form; the cross terms give a $\sin 2\vartheta$.

**Solution.**

*Step 1.* $Ax_0 = (2\cos\vartheta + \sin\vartheta, \ \cos\vartheta + 2\sin\vartheta)^{\top}$.

*Step 2.* $x_0^{\top}Ax_0 = 2\cos^{2}\vartheta + 2\sin^{2}\vartheta + 2\sin\vartheta\cos\vartheta
= 2 + \sin 2\vartheta$, and $x_0^{\top}x_0 = 1$.

*Step 3.* At $\vartheta = 0.1$: $\mu_0 = 2 + \sin 0.2$.

$$
\boxed{\mu_0 = 2 + \sin 2\vartheta = 2 + \sin(0.2) \approx 2.198669}
$$

**Key takeaway.** $\mu_0$ ranges over $[1, 3]$ as $\vartheta$ turns, hitting the eigenvalue $3$
exactly at $\vartheta = \pi/4$, where $x_0$ is the eigenvector — Rayleigh quotients are stationary
there (Problem L1.8).

In [9]:
A = np.array([[2.0, 1.0], [1.0, 2.0]])
for th in (0.1, np.pi / 4):
    x = np.array([np.cos(th), np.sin(th)])
    print(f"theta = {th:.6f}   R_A(x) = {x @ A @ x:.10f}   2 + sin(2 theta) = {2 + np.sin(2*th):.10f}")
    assert abs(x @ A @ x - (2 + np.sin(2 * th))) < 1e-12

theta = 0.100000   R_A(x) = 2.1986693308   2 + sin(2 theta) = 2.1986693308
theta = 0.785398   R_A(x) = 3.0000000000   2 + sin(2 theta) = 3.0000000000


## L1 — Foundations

### Problem L1.1 — Unitary similarity preserves the spectrum

**Statement.** Let $Q \in \mathbb{C}^{n \times n}$ be unitary and $B = Q^{\ast}AQ$. Prove that
$A$ and $B$ have the same characteristic polynomial, hence the same spectrum with
multiplicities.

**Intuition.** A unitary similarity is a change of orthonormal basis; eigenvalues belong to the
operator, not to the basis.

**Solution.**

*Step 1.* Write $\lambda I = Q^{\ast}(\lambda I)Q$ and factor:

$$
B - \lambda I = Q^{\ast}AQ - Q^{\ast}(\lambda I)Q = Q^{\ast}(A - \lambda I)Q .
$$

*Step 2.* Determinants multiply, so
$\det(B - \lambda I) = \det(Q^{\ast})\det(A - \lambda I)\det(Q)$.

*Step 3.* $\det(Q^{\ast})\det(Q) = \det(Q^{\ast}Q) = \det(I) = 1$.

$$
\boxed{\det(B - \lambda I) = \det(A - \lambda I)}
$$

**Key takeaway.** This is the licence for every algorithm in the module: each QR step, each
Householder reflection, each Givens rotation is a unitary similarity, so the spectrum is carried
along untouched while the shape improves.

In [10]:
M = rng.standard_normal((5, 5))
Q5, _ = np.linalg.qr(rng.standard_normal((5, 5)))
B = Q5.T @ M @ Q5
print("char. poly of M:", np.poly(M))
print("char. poly of B:", np.poly(B))
print("max coefficient difference:", np.abs(np.poly(M) - np.poly(B)).max())
assert np.abs(np.poly(M) - np.poly(B)).max() < 1e-10

char. poly of M: [ 1.      3.957   6.6442 18.9609 34.9387 22.2606]
char. poly of B: [ 1.      3.957   6.6442 18.9609 34.9387 22.2606]
max coefficient difference: 1.7763568394002505e-14


### Problem L1.2 — One step of power iteration

**Statement.** For $A = \left(\begin{smallmatrix}4 & 1 \\ 2 & 3\end{smallmatrix}\right)$ and
$x_0 = (1,1)^{\top}$, compute the normalized iterate $x_1$ and the Rayleigh quotient estimate
$\lambda^{(1)} = x_1^{\top}Ax_1$.

**Intuition.** Multiply, then normalize; the Rayleigh quotient reads off the eigenvalue.

**Solution.**

*Step 1.* $Ax_0 = (4+1, \ 2+3)^{\top} = (5,5)^{\top}$.

*Step 2.* $\lVert (5,5) \rVert = 5\sqrt2$, so $x_1 = \tfrac{1}{\sqrt2}(1,1)^{\top}$.

*Step 3.* $Ax_1 = \tfrac{5}{\sqrt2}(1,1)^{\top}$, so $x_1^{\top}Ax_1 = 5$.

$$
\boxed{x_1 = \tfrac{1}{\sqrt2}(1,1)^{\top}, \qquad \lambda^{(1)} = 5}
$$

**Key takeaway.** The iteration converged in one step because $x_0$ was already an eigenvector:
$A(1,1)^{\top} = 5(1,1)^{\top}$. The other eigenvalue is $2$, so a generic start would converge at
rate $2/5$.

In [11]:
A = np.array([[4.0, 1.0], [2.0, 3.0]])
x0 = np.array([1.0, 1.0])
x1 = A @ x0
x1 = x1 / np.linalg.norm(x1)
print("x1 =", x1, " (1,1)/sqrt2 =", np.array([1.0, 1.0]) / np.sqrt(2))
print("Rayleigh quotient =", x1 @ A @ x1)
print("spectrum of A:", np.sort(np.linalg.eigvals(A).real))
assert np.allclose(x1, np.array([1.0, 1.0]) / np.sqrt(2)) and abs(x1 @ A @ x1 - 5.0) < 1e-12

x1 = [0.7071 0.7071]  (1,1)/sqrt2 = [0.7071 0.7071]
Rayleigh quotient = 4.999999999999999
spectrum of A: [2. 5.]


### Problem L1.3 — One step of shifted inverse iteration

**Statement.** With $A = \left(\begin{smallmatrix}3 & 1 \\ 1 & 3\end{smallmatrix}\right)$,
$\mu = 1$ and $x_0 = (1,0)^{\top}$, compute the normalized $x_1$.

**Intuition.** Solve $(A - \mu I)y = x_0$ and normalize; the shift $\mu = 1$ is nearer the
eigenvalue $2$ than the eigenvalue $4$.

**Solution.**

*Step 1.* $A - I = \left(\begin{smallmatrix}2 & 1 \\ 1 & 2\end{smallmatrix}\right)$, with
$\det = 3$.

*Step 2.* $(A-I)^{-1} = \tfrac13\left(\begin{smallmatrix}2 & -1 \\ -1 & 2\end{smallmatrix}\right)$,
so $y_1 = \tfrac13(2, -1)^{\top}$.

*Step 3.* $\lVert y_1 \rVert = \sqrt5/3$, hence $x_1 = \tfrac{1}{\sqrt5}(2,-1)^{\top}$.

$$
\boxed{x_1 = \tfrac{1}{\sqrt5}(2,-1)^{\top}}
$$

**Key takeaway.** $x_1$ leans towards $(1,-1)^{\top}$, the eigenvector for $\lambda = 2$ — the
eigenvalue nearest the shift, exactly as Theorem 4.1 predicts, at rate
$\lvert 2-1 \rvert / \lvert 4-1 \rvert = 1/3$.

In [12]:
A = np.array([[3.0, 1.0], [1.0, 3.0]])
y = np.linalg.solve(A - np.eye(2), np.array([1.0, 0.0]))
x1 = y / np.linalg.norm(y)
print("y1 =", y, "  x1 =", x1, "  (2,-1)/sqrt5 =", np.array([2.0, -1.0]) / np.sqrt(5))
x = np.array([1.0, 0.0])
for _ in range(12):
    y = np.linalg.solve(A - np.eye(2), x)
    x = y / np.linalg.norm(y)
print("after 12 steps:", x, "  Rayleigh quotient:", x @ A @ x, "  (target eigenvalue 2)")
assert np.allclose(x1, np.array([2.0, -1.0]) / np.sqrt(5))
assert abs(x @ A @ x - 2.0) < 1e-8

y1 = [ 0.6667 -0.3333]   x1 = [ 0.8944 -0.4472]   (2,-1)/sqrt5 = [ 0.8944 -0.4472]
after 12 steps: [ 0.7071 -0.7071]   Rayleigh quotient: 2.000000000007081   (target eigenvalue 2)


### Problem L1.4 — One unshifted QR step

**Statement.** For $A_0 = \left(\begin{smallmatrix}0 & 2 \\ 1 & 0\end{smallmatrix}\right)$ compute
$A_1 = R_0Q_0$, and say what happens on further steps.

**Intuition.** Factor, then multiply the factors back in the other order.

**Solution.**

*Step 1.* The first column is $(0,1)^{\top}$, already of unit length, so
$q_1 = (0,1)^{\top}$; the second column $(2,0)^{\top}$ is orthogonal to it, so
$q_2 = (1,0)^{\top}$ and
$Q_0 = \left(\begin{smallmatrix}0 & 1 \\ 1 & 0\end{smallmatrix}\right)$.

*Step 2.* $R_0 = Q_0^{\top}A_0 = \left(\begin{smallmatrix}1 & 0 \\ 0 & 2\end{smallmatrix}\right)$,
upper triangular with positive diagonal.

*Step 3.* $A_1 = R_0Q_0 = \left(\begin{smallmatrix}0 & 1 \\ 2 & 0\end{smallmatrix}\right)$.

*Step 4.* Repeating the argument on $A_1$ returns $A_2 = A_0$. The iteration cycles with period
two and never converges.

$$
\boxed{A_1 = \begin{pmatrix} 0 & 1 \\ 2 & 0 \end{pmatrix}, \quad A_2 = A_0, \quad \text{no convergence}}
$$

**Key takeaway.** The eigenvalues are $\pm\sqrt2$, of **equal modulus**, so the hypothesis
$\lvert \lambda_1 \rvert \gt \lvert \lambda_2 \rvert$ of Theorem 4.4 fails and the conclusion
fails with it. A shift repairs it, as Section 7.3 of the theory notebook shows on the closely
related matrix $\left(\begin{smallmatrix}0 & 1 \\ 1 & 0\end{smallmatrix}\right)$.

In [13]:
A0 = np.array([[0.0, 2.0], [1.0, 0.0]])
Ak = A0.copy()
for k in range(4):
    print(f"  A_{k} = {Ak.ravel()}")
    Q, R = qr_positive(Ak)
    Ak = R @ Q
print("eigenvalues:", np.sort(np.linalg.eigvals(A0).real), "  moduli:", np.abs(np.linalg.eigvals(A0)))
Q, R = qr_positive(A0)
assert np.allclose(R @ Q, [[0.0, 1.0], [2.0, 0.0]])
assert np.allclose(Ak, A0)   # period two: A_4 = A_0

  A_0 = [0. 2. 1. 0.]
  A_1 = [0. 1. 2. 0.]
  A_2 = [0. 2. 1. 0.]
  A_3 = [0. 1. 2. 0.]
eigenvalues: [-1.4142  1.4142]   moduli: [1.4142 1.4142]


### Problem L1.5 — A Householder reflector for Hessenberg reduction

**Statement.** Build the $3 \times 3$ Householder matrix $H_1$ that annihilates the entry below
the subdiagonal in the first column of
$A = \left(\begin{smallmatrix}1 & 2 & 3 \\ 3 & 1 & 4 \\ 4 & 2 & 1\end{smallmatrix}\right)$.

**Intuition.** Reflect the sub-column $(3,4)^{\top}$ onto the first axis, leaving row and column
one untouched so the similarity does not undo the work.

**Solution.**

*Step 1.* The sub-column is $x = (3,4)^{\top}$ with $\lVert x \rVert = 5$.

*Step 2.* Choose $v = x + \operatorname{sgn}(x_1)\lVert x \rVert e_1 = (8,4)^{\top} \propto (2,1)^{\top}$,
the sign chosen to avoid cancellation.

*Step 3.* $P = I - 2vv^{\top}/(v^{\top}v)$ with $v^{\top}v = 5$ gives

$$
P = \begin{pmatrix} 1 - 8/5 & -4/5 \\ -4/5 & 1 - 2/5 \end{pmatrix}
= \begin{pmatrix} -3/5 & -4/5 \\ -4/5 & 3/5 \end{pmatrix},
\qquad Px = (-5, 0)^{\top} .
$$

*Step 4.* Embed with a leading $1$.

$$
\boxed{H_1 = \begin{pmatrix} 1 & 0 & 0 \\ 0 & -3/5 & -4/5 \\ 0 & -4/5 & 3/5 \end{pmatrix}}
$$

**Key takeaway.** The reflector acts on rows $2,3$ only, so the similarity $H_1AH_1$ preserves the
zero it created. Reducing to Hessenberg rather than to triangular form is exactly what makes that
possible.

In [14]:
A = np.array([[1.0, 2.0, 3.0], [3.0, 1.0, 4.0], [4.0, 2.0, 1.0]])
x = A[1:, 0]
v = np.array([2.0, 1.0])
P = np.eye(2) - 2.0 * np.outer(v, v) / (v @ v)
H1 = np.eye(3)
H1[1:, 1:] = P
print("P =\n", P, "\nP x =", P @ x)
Hs = H1 @ A @ H1
print("H1 A H1 =\n", Hs)
print("entry (3,1) after the similarity:", Hs[2, 0])
print("spectrum preserved:", np.sort(np.linalg.eigvals(A).real), np.sort(np.linalg.eigvals(Hs).real))
assert np.allclose(P, [[-0.6, -0.8], [-0.8, 0.6]])
assert abs(Hs[2, 0]) < 1e-12
assert np.allclose(np.sort(np.linalg.eigvals(A).real), np.sort(np.linalg.eigvals(Hs).real))

P =
 [[-0.6 -0.8]
 [-0.8  0.6]] 
P x = [-5. -0.]
H1 A H1 =
 [[ 1.   -3.6   0.2 ]
 [-5.    3.88 -0.16]
 [-0.    1.84 -1.88]]
entry (3,1) after the similarity: -4.440892098500626e-16
spectrum preserved: [-2.2788 -1.5958  6.8746] [-2.2788 -1.5958  6.8746]


### Problem L1.6 — Flop count of the Hessenberg reduction

**Statement.** Householder reduction takes a dense $A \in \mathbb{R}^{n \times n}$ to upper
Hessenberg form $H = Q^{\top}AQ$. Give the leading coefficient $C$ in $Cn^{3} + O(n^{2})$ flops,
counting the similarity but not the accumulation of $Q$.

**Intuition.** Each step reflects a trailing block from the left and the full matrix from the
right; sum the two costs over the $n-2$ steps.

**Solution.**

*Step 1.* At step $k$ the reflector has length $n-k$.

*Step 2.* Applying it from the left updates an $(n-k) \times (n-k+1)$ block:
$\approx 4(n-k)^{2}$ flops.

*Step 3.* Applying it from the right updates an $n \times (n-k)$ block:
$\approx 4n(n-k)$ flops.

*Step 4.* Sum and replace by an integral:

$$
\sum_{k=1}^{n-2} \bigl[ 4(n-k)^{2} + 4n(n-k) \bigr]
\approx \int_0^n \bigl( 4(n-x)^{2} + 4n(n-x) \bigr)\,dx
= \frac{4}{3}n^{3} + 2n^{3} .
$$

$$
\boxed{C = \frac{10}{3}, \qquad \tfrac{10}{3}n^{3} \text{ flops}}
$$

**Key takeaway.** A one-off $\tfrac{10}{3}n^{3}$ buys $O(n^{2})$ per QR step instead of
$O(n^{3})$ (Theorem 4.5). For symmetric matrices the reduction is to tridiagonal form and costs
$\tfrac43 n^{3}$, and each step then costs $O(n)$.

In [15]:
def hessenberg_flops(n):
    """Leading-order flop count summed exactly over the n-2 Householder steps."""
    return sum(4.0 * (n - k) ** 2 + 4.0 * n * (n - k) for k in range(1, n - 1))


for n in (50, 200, 800, 3200):
    print(f"n = {n:5d}   counted / n^3 = {hessenberg_flops(n)/n**3:.6f}   10/3 = {10/3:.6f}")
assert abs(hessenberg_flops(3200) / 3200 ** 3 - 10 / 3) < 5e-3

n =    50   counted / n^3 = 3.251968   10/3 = 3.333333
n =   200   counted / n^3 = 3.313249   10/3 = 3.333333
n =   800   counted / n^3 = 3.328328   10/3 = 3.333333
n =  3200   counted / n^3 = 3.332083   10/3 = 3.333333


### Problem L1.7 — The first Lanczos step

**Statement.** For $A = \left(\begin{smallmatrix}2 & 1 \\ 1 & 2\end{smallmatrix}\right)$ and
$b = (1,0)^{\top}$, compute $q_1$, $\alpha_1$, $\beta_1$ and $q_2$.

**Intuition.** Normalize $b$, project $Aq_1$ onto $q_1$, and normalize the remainder.

**Solution.**

*Step 1.* $q_1 = (1,0)^{\top}$.

*Step 2.* $Aq_1 = (2,1)^{\top}$, so $\alpha_1 = q_1^{\top}Aq_1 = 2$.

*Step 3.* $r_1 = Aq_1 - \alpha_1 q_1 = (0,1)^{\top}$, so $\beta_1 = 1$.

*Step 4.* $q_2 = (0,1)^{\top}$.

$$
\boxed{\alpha_1 = 2, \quad \beta_1 = 1, \quad q_2 = (0,1)^{\top}}
$$

**Key takeaway.** After two steps $T_2 = A$ itself, because $\mathcal{K}_2(A, e_1) = \mathbb{R}^2$:
Lanczos on an $n \times n$ matrix run to $n$ steps is just a tridiagonalization, and its value lies
in stopping early.

In [16]:
A = np.array([[2.0, 1.0], [1.0, 2.0]])
a, bt, Qb = lanczos(A, np.array([1.0, 0.0]), 2)
print("alpha =", a, "  beta =", bt[:1], "  q2 =", Qb[:, 1])
T2 = np.diag(a) + np.diag(bt[:1], 1) + np.diag(bt[:1], -1)
print("T2 =\n", T2, "\nequal to A:", np.allclose(T2, A))
assert abs(a[0] - 2.0) < 1e-12 and abs(bt[0] - 1.0) < 1e-12
assert np.allclose(np.abs(Qb[:, 1]), [0.0, 1.0])

alpha = [2. 2.]   beta = [1.]   q2 = [0. 1.]
T2 =
 [[2. 1.]
 [1. 2.]] 
equal to A: True


### Problem L1.8 — Eigenvectors are the stationary points of the Rayleigh quotient

**Statement.** For symmetric $A$ and $x \neq 0$, show
$\nabla R_A(x) = \tfrac{2}{x^{\top}x}\bigl( Ax - R_A(x)x \bigr)$, and deduce that
$\nabla R_A(x) = 0$ if and only if $x$ is an eigenvector.

**Intuition.** The quotient rule, with $\nabla(x^{\top}Ax) = 2Ax$ available only because
$A = A^{\top}$.

**Solution.**

*Step 1.* Put $f(x) = x^{\top}Ax$ and $g(x) = x^{\top}x$. Symmetry gives $\nabla f = 2Ax$, and
$\nabla g = 2x$.

*Step 2.* The quotient rule gives

$$
\nabla R_A(x) = \frac{(\nabla f) g - f (\nabla g)}{g^{2}}
= \frac{2Ax\,(x^{\top}x) - (x^{\top}Ax)\,2x}{(x^{\top}x)^{2}} .
$$

*Step 3.* Factor out $2/(x^{\top}x)$:

$$
\nabla R_A(x) = \frac{2}{x^{\top}x}\left( Ax - \frac{x^{\top}Ax}{x^{\top}x}\,x \right) .
$$

*Step 4.* This vanishes exactly when $Ax = R_A(x)\,x$, that is when $x$ is an eigenvector with
eigenvalue $R_A(x)$.

$$
\boxed{\nabla R_A(x) = \frac{2}{x^{\top}x}\bigl( Ax - R_A(x)x \bigr) = 0 \iff Ax = R_A(x)x}
$$

**Key takeaway.** Stationarity is why the Rayleigh quotient is second-order accurate
(Problem L1.9) and therefore why RQI is cubic (Theorem 4.2). Note also that
$\nabla R_A(x)$ is the residual up to scale, so the gradient is free once the residual is known.

In [17]:
M = rng.standard_normal((4, 4))
A = (M + M.T) / 2.0


def rayleigh(x):
    return x @ A @ x / (x @ x)


def grad_rayleigh(x):
    return 2.0 / (x @ x) * (A @ x - rayleigh(x) * x)


x = rng.standard_normal(4)
h = 1e-6
num = np.array([(rayleigh(x + h * e) - rayleigh(x - h * e)) / (2 * h) for e in np.eye(4)])
print("analytic gradient:", grad_rayleigh(x))
print("finite difference:", num)
print("max difference   :", np.abs(num - grad_rayleigh(x)).max())
lam, V = np.linalg.eigh(A)
print("gradient at an eigenvector:", grad_rayleigh(V[:, 2]), " norm",
      np.linalg.norm(grad_rayleigh(V[:, 2])))
assert np.abs(num - grad_rayleigh(x)).max() < 1e-7
assert np.linalg.norm(grad_rayleigh(V[:, 2])) < 1e-12

analytic gradient: [ 0.1561 -0.0046 -0.3685  0.0432]
finite difference: [ 0.1561 -0.0046 -0.3685  0.0432]
max difference   : 9.459130700939511e-11
gradient at an eigenvector: [ 0.  0. -0. -0.]  norm 1.0532500405730101e-15


### Problem L1.9 — Quadratic accuracy of the Rayleigh quotient

**Statement.** Let $A$ be symmetric, $Av_1 = \lambda_1 v_1$ with $\lVert v_1 \rVert = 1$, and
$x = v_1 + \epsilon e$ with $e \perp v_1$, $\lVert e \rVert = 1$. Show
$\lvert R_A(x) - \lambda_1 \rvert = O(\epsilon^{2})$ and identify the constant.

**Intuition.** The first-order term vanishes because $e^{\top}Av_1 = \lambda_1 e^{\top}v_1 = 0$.

**Solution.**

*Step 1.* Numerator:

$$
x^{\top}Ax = \lambda_1 + 2\epsilon\, e^{\top}Av_1 + \epsilon^{2}\, e^{\top}Ae
= \lambda_1 + \epsilon^{2}\, e^{\top}Ae ,
$$

because $e^{\top}Av_1 = \lambda_1 e^{\top}v_1 = 0$.

*Step 2.* Denominator: $x^{\top}x = 1 + \epsilon^{2}$, since $v_1^{\top}e = 0$.

*Step 3.* Divide and expand $(1+\epsilon^{2})^{-1} = 1 - \epsilon^{2} + O(\epsilon^{4})$:

$$
R_A(x) = \lambda_1 + \epsilon^{2}\bigl( e^{\top}Ae - \lambda_1 \bigr) + O(\epsilon^{4}) .
$$

$$
\boxed{R_A(x) - \lambda_1 = \epsilon^{2}\bigl( R_A(e) - \lambda_1 \bigr) + O(\epsilon^{4})}
$$

**Key takeaway.** The constant is the gap between $\lambda_1$ and the Rayleigh quotient of the
error direction, bounded by the spread $\lambda_1 - \lambda_n$. This is Step 1 of Proof 5.2 in
perturbation form.

In [18]:
M = rng.standard_normal((5, 5))
A = (M + M.T) / 2.0
lam, V = np.linalg.eigh(A)
v1 = V[:, -1]
e = V[:, 0]
lam1 = lam[-1]
print("  eps        |R(x) - lambda_1|      eps^2 |R(e) - lambda_1|       ratio")
for eps in (1e-1, 1e-2, 1e-3, 1e-4):
    x = v1 + eps * e
    err = abs(x @ A @ x / (x @ x) - lam1)
    pred = eps ** 2 * abs(e @ A @ e - lam1)
    print(f"  {eps:.0e}     {err:.6e}        {pred:.6e}          {err/pred:.6f}")
    assert abs(err / pred - 1.0) < 0.05

  eps        |R(x) - lambda_1|      eps^2 |R(e) - lambda_1|       ratio
  1e-01     4.383209e-02        4.427041e-02          0.990099
  1e-02     4.426598e-04        4.427041e-04          0.999900
  1e-03     4.427036e-06        4.427041e-06          0.999999
  1e-04     4.427041e-08        4.427041e-08          1.000000


### Problem L1.10 — A QR step preserves the Hessenberg shape

**Statement.** Let $H$ be upper Hessenberg with QR factorization $H = QR$ obtained from Givens
rotations. Show that $H_{+} = RQ$ is upper Hessenberg.

**Intuition.** $Q$ inherits the Hessenberg shape from the rotations, and upper triangular times
upper Hessenberg is upper Hessenberg.

**Solution.**

*Step 1.* $R = G_{n-1}\cdots G_1 H$ where $G_i$ acts on rows $i, i+1$, so
$Q = G_1^{\top}\cdots G_{n-1}^{\top}$.

*Step 2.* Each partial product $G_1^{\top}\cdots G_m^{\top}$ is upper Hessenberg and equals the
identity in columns $m+2, \dots, n$ (Proof 5.5), so $Q$ is upper Hessenberg.

*Step 3.* Fix $i \gt j+1$ and expand $(RQ)_{ij} = \sum_k R_{ik}Q_{kj}$. A non-zero term needs
$k \ge i$ (triangular $R$) and $k \le j+1$ (Hessenberg $Q$), so $i \le k \le j+1 \lt i$ — a
contradiction.

$$
\boxed{(RQ)_{ij} = 0 \text{ for } i \gt j+1}
$$

**Key takeaway.** Structure is preserved for free, so the $\tfrac{10}{3}n^{3}$ reduction of
Problem L1.6 is paid once and every subsequent step costs $O(n^{2})$.

In [19]:
Hm = np.triu(rng.standard_normal((6, 6)), -1)
Ak = Hm.copy()
for k in range(6):
    below = np.abs(np.tril(Ak, -2)).max()
    print(f"  step {k}: largest entry below the subdiagonal = {below:.3e}")
    assert below < 1e-12
    Q, R = qr_positive(Ak)
    print(f"          Q below-subdiagonal max = {np.abs(np.tril(Q, -2)).max():.3e}")
    assert np.abs(np.tril(Q, -2)).max() < 1e-12
    Ak = R @ Q

  step 0: largest entry below the subdiagonal = 0.000e+00
          Q below-subdiagonal max = 0.000e+00
  step 1: largest entry below the subdiagonal = 0.000e+00
          Q below-subdiagonal max = 0.000e+00
  step 2: largest entry below the subdiagonal = 0.000e+00
          Q below-subdiagonal max = 0.000e+00
  step 3: largest entry below the subdiagonal = 0.000e+00
          Q below-subdiagonal max = 0.000e+00
  step 4: largest entry below the subdiagonal = 0.000e+00
          Q below-subdiagonal max = 0.000e+00
  step 5: largest entry below the subdiagonal = 0.000e+00
          Q below-subdiagonal max = 0.000e+00


### Problem L1.11 — Dimension of a Krylov subspace

**Statement.** Let $d$ be the degree of the minimal polynomial of $b \neq 0$ with respect to $A$,
that is the least $d$ with $p(A)b = 0$ for some monic $p$ of degree $d$. Show
$\dim \mathcal{K}_m(A,b) = \min(m, d)$; in particular $\dim \mathcal{K}_m(A,b) = m$ exactly when
$m \le d$.

**Intuition.** The vectors $b, Ab, A^{2}b, \dots$ stay independent until the first linear relation,
and that relation is the minimal polynomial.

**Solution.**

*Step 1.* If $\sum_{i=0}^{r} c_i A^{i}b = 0$ with $c_r \neq 0$ and $r \lt d$, dividing by $c_r$
produces a monic annihilating polynomial of degree $r \lt d$, contradicting minimality. So
$b, Ab, \dots, A^{d-1}b$ are linearly independent.

*Step 2.* Hence for $m \le d$ the set $\lbrace b, \dots, A^{m-1}b \rbrace$ is a subset of an
independent set and $\dim\mathcal{K}_m = m$.

*Step 3.* From $p(A)b = 0$ we get
$A^{d}b = -\sum_{i \lt d} c_i A^{i}b \in \mathcal{K}_d$. Multiplying by $A$ and inducting,
$A^{k}b \in \mathcal{K}_d$ for every $k \ge d$, so $\mathcal{K}_m = \mathcal{K}_d$ for $m \ge d$.

$$
\boxed{\dim \mathcal{K}_m(A,b) = \min(m, d)}
$$

**Key takeaway.** Breakdown in Arnoldi or Lanczos happens at step $d$ and not before, and it is
*good* news: $\mathcal{K}_d$ is invariant, so the Ritz values are exact eigenvalues. Example 6.5
is the case $n = 3$, $d = 2$.

In [20]:
lamK = np.array([4.0, 2.0, 2.0, 1.0])
Qk, _ = np.linalg.qr(rng.standard_normal((4, 4)))
A = Qk @ np.diag(lamK) @ Qk.T
b_full = Qk @ np.array([1.0, 1.0, 0.0, 1.0])       # misses one eigendirection of the pair
b_short = Qk @ np.array([1.0, 1.0, 0.0, 0.0])      # only two eigendirections
for name, b in (("b with 3 distinct eigenvalues", b_full), ("b with 2", b_short)):
    K = np.column_stack([np.linalg.matrix_power(A, i) @ b for i in range(4)])
    dims = [np.linalg.matrix_rank(K[:, :m]) for m in range(1, 5)]
    print(f"  {name}: dim K_m for m = 1..4 -> {dims}")
assert [np.linalg.matrix_rank(np.column_stack(
    [np.linalg.matrix_power(A, i) @ b_short for i in range(4)])[:, :m]) for m in range(1, 5)] == [1, 2, 2, 2]

  b with 3 distinct eigenvalues: dim K_m for m = 1..4 -> [np.int64(1), np.int64(2), np.int64(3), np.int64(3)]
  b with 2: dim K_m for m = 1..4 -> [np.int64(1), np.int64(2), np.int64(2), np.int64(2)]


### Problem L1.12 — Spectrum of a Householder reflector

**Statement.** For $v \neq 0$ let $H = I - 2vv^{\top}/(v^{\top}v)$. Find all eigenvalues of $H$
with their multiplicities, and $\det H$.

**Intuition.** $H$ reflects across the hyperplane $v^{\perp}$: it negates $v$ and fixes everything
orthogonal to it.

**Solution.**

*Step 1.* $Hv = v - 2v(v^{\top}v)/(v^{\top}v) = -v$, so $-1$ is an eigenvalue.

*Step 2.* If $v^{\top}w = 0$ then $Hw = w$, and $v^{\perp}$ has dimension $n-1$, so $+1$ has
geometric multiplicity at least $n-1$.

*Step 3.* The two eigenspaces are orthogonal and their dimensions sum to $n$, so the list is
complete and $H$ is symmetric orthogonal.

$$
\boxed{\operatorname{spec}(H) = \lbrace -1 \rbrace \cup \lbrace +1 \rbrace^{\,n-1},
\qquad \det H = -1}
$$

**Key takeaway.** $\det H = -1$ means a reflector is not a rotation; a Givens rotation has
$\det = +1$. Both are orthogonal, so both preserve the spectrum of $A$ under similarity
(Problem L1.1), which is all the algorithms need.

In [21]:
v = rng.standard_normal(6)
H = np.eye(6) - 2.0 * np.outer(v, v) / (v @ v)
ev = np.linalg.eigvalsh(H)
print("eigenvalues:", np.sort(ev))
print("det =", np.linalg.det(H), "   H^T H - I:", np.abs(H.T @ H - np.eye(6)).max())
assert np.allclose(np.sort(ev), [-1.0] + [1.0] * 5)
assert abs(np.linalg.det(H) + 1.0) < 1e-12

eigenvalues: [-1.  1.  1.  1.  1.  1.]
det = -1.0000000000000004    H^T H - I: 2.220446049250313e-16


## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — PageRank: the damping factor manufactures a spectral gap

**Statement.** Let $P$ be column-stochastic and
$M = dP + \tfrac{1-d}{n}\mathbf{1}\mathbf{1}^{\top}$ with $d \in (0,1)$. Prove that $1$ is an
eigenvalue of $M$ and every other eigenvalue satisfies $\lvert \lambda \rvert \le d$. Then find the
number of power-iteration steps that guarantees a $10^{-6}$ error reduction at $d = 0.85$.

**Intuition.** The rank-one teleportation term acts as the identity on the constant direction and
as zero on everything orthogonal to $\mathbf{1}$, so it fixes the Perron value and shrinks the
rest by $d$.

**Solution.**

*Step 1.* $\mathbf{1}^{\top}P = \mathbf{1}^{\top}$ and $\mathbf{1}^{\top}\mathbf{1} = n$, so

$$
\mathbf{1}^{\top}M = d\,\mathbf{1}^{\top} + \frac{1-d}{n}\,n\,\mathbf{1}^{\top} = \mathbf{1}^{\top} .
$$

Hence $M$ is column-stochastic and $1 \in \operatorname{spec}(M)$.

*Step 2.* Let $Pv = \lambda v$ with $\lambda \neq 1$. Because left and right eigenvectors for
distinct eigenvalues are biorthogonal and $\mathbf{1}^{\top}$ is a left eigenvector for $1$, we get
$\mathbf{1}^{\top}v = 0$. Therefore

$$
Mv = dPv + \frac{1-d}{n}\mathbf{1}\,(\mathbf{1}^{\top}v) = d\lambda v .
$$

*Step 3.* $P$ is column-stochastic so $\rho(P) = 1$, giving $\lvert d\lambda \rvert \le d$.

*Step 4.* Power iteration contracts by at most $\lvert \lambda_2(M) \rvert \le d$ per step, so
$d^{k} \le 10^{-6}$ requires

$$
k \ \ge\ \frac{6\ln 10}{-\ln d} = \frac{13.8155}{0.16252} = 85.01 .
$$

$$
\boxed{\lvert \lambda_2(M) \rvert \le d, \qquad k = 86 \ \text{ iterations at } d = 0.85}
$$

**Key takeaway.** The bound is independent of $n$: the same $86$ iterations serve a six-page graph
and a billion-page one. Damping is not a modelling choice, it is a guaranteed spectral gap of
$1 - d$.

In [22]:
links = {0: [1, 2], 1: [2], 2: [0, 3], 3: [4], 4: [3, 5], 5: [0, 3, 4]}
npages = 6
P = np.zeros((npages, npages))
for src, outs in links.items():
    for dst in outs:
        P[dst, src] = 1.0 / len(outs)
d = 0.85
Mg = d * P + (1.0 - d) / npages * np.ones((npages, npages))
ev = np.linalg.eigvals(Mg)
ev = ev[np.argsort(-np.abs(ev))]
print("column sums:", Mg.sum(axis=0))
print("eigenvalues by modulus:", np.round(np.abs(ev), 6))
print(f"|lambda_2| = {abs(ev[1]):.6f}  <=  d = {d}")
k_needed = int(np.ceil(6 * np.log(10) / (-np.log(d))))
print("guaranteed iterations for 1e-6:", k_needed)
wg, Vg = np.linalg.eig(Mg)
pi = np.real(Vg[:, int(np.argmin(np.abs(wg - 1.0)))])
pi /= pi.sum()
x = np.ones(npages) / npages
for _ in range(k_needed):
    x = Mg @ x
print("error after 86 steps:", np.linalg.norm(x - pi))
assert abs(ev[0] - 1.0) < 1e-12 and abs(ev[1]) <= d + 1e-12
assert k_needed == 86 and np.linalg.norm(x - pi) < 1e-6

column sums: [1. 1. 1. 1. 1. 1.]
eigenvalues by modulus: [1.     0.5835 0.5431 0.5431 0.5055 0.    ]
|lambda_2| = 0.583494  <=  d = 0.85
guaranteed iterations for 1e-6: 86
error after 86 steps: 1.1944575768113593e-15


### Problem L2.2 — The Hessian spectrum of a network, without the Hessian

**Statement.** A model has $p = 10^{8}$ parameters and empirical risk $J(\theta)$. Derive
Pearlmutter's identity for the Hessian-vector product $\nabla^{2}J(\theta)\,v$ from two automatic
differentiation passes, state its memory cost, and explain how Lanczos then returns the extreme
curvatures.

**Intuition.** Lanczos never needs the matrix, only its action; and the action of a Hessian is a
directional derivative of the gradient, which forward-mode differentiation computes exactly.

**Solution.**

*Step 1 — the memory barrier.* Storing $\nabla^{2}J$ needs $p^{2} = 10^{16}$ numbers, about
$4 \times 10^{4}$ terabytes in float32. It cannot be formed.

*Step 2 — the identity.* Define $g(\theta) = \nabla J(\theta)$ and, for fixed $v$, the curve
$r \mapsto \theta + rv$. The chain rule gives

$$
\left. \frac{d}{dr} g(\theta + rv) \right\rvert_{r=0} = \nabla^{2}J(\theta)\, v .
$$

*Step 3 — two passes.* One reverse-mode pass evaluates $g(\theta)$ as a computational graph. Applying
forward-mode differentiation to that graph in the direction $v$ evaluates the derivative above
**exactly**, not by finite differences: the tangent variable of every node is propagated alongside
its value.

*Step 4 — cost.* The tangent computation doubles the node count, so the whole product costs about
twice a gradient in time and $O(p)$ in memory. No $p \times p$ object is ever created.

*Step 5 — Lanczos on top.* Theorem 4.7 needs only $v \mapsto \nabla^{2}J\,v$. After $m$ steps the
tridiagonal $T_m$ is $m \times m$ and its extreme Ritz values approximate
$\lambda_{\max}$ and $\lambda_{\min}$ of the Hessian, at a total cost of $m$ Hessian-vector
products and $O(mp)$ memory.

$$
\boxed{\nabla^{2}J(\theta)\,v = \left. \tfrac{\partial}{\partial r}\, \nabla J(\theta + rv) \right\rvert_{r=0},
\qquad \text{memory } O(p), \qquad \text{time } \approx 2 \times \text{one gradient}}
$$

**Key takeaway.** $\lambda_{\max}$ is the smoothness constant $L$ that caps a stable step size at
$\eta \lt 2/L$, and the near-zero eigenvalues are the flat directions of the minimum. Theorem 4.9
is the warning: run this without reorthogonalization and $\lambda_{\max}$ will be reported several
times over.

In [23]:
# A small but genuine example: least squares with a tanh feature map, so J is nonlinear
# in theta and its Hessian is dense. The Hessian-vector product is evaluated by the exact
# forward-over-reverse identity, implemented in closed form, and checked against the dense Hessian.
nfeat, nobs = 30, 120
Xd = rng.standard_normal((nobs, nfeat))
yd = rng.standard_normal(nobs)


def residual(theta):
    return np.tanh(Xd @ theta) - yd


def grad_J(theta):
    a = Xd @ theta
    return Xd.T @ ((np.tanh(a) - yd) * (1.0 - np.tanh(a) ** 2))


def hess_J(theta):
    a = Xd @ theta
    t = np.tanh(a)
    w = (1.0 - t ** 2) ** 2 - 2.0 * t * (1.0 - t ** 2) * (t - yd)
    return Xd.T @ (w[:, None] * Xd)


def hvp(theta, v):
    """Exact Hessian-vector product by differentiating the gradient along v."""
    a = Xd @ theta
    t = np.tanh(a)
    da = Xd @ v
    w = (1.0 - t ** 2) ** 2 - 2.0 * t * (1.0 - t ** 2) * (t - yd)
    return Xd.T @ (w * da)


theta0 = 0.3 * rng.standard_normal(nfeat)
v = rng.standard_normal(nfeat)
H = hess_J(theta0)
print("||H v - hvp(v)|| / ||H v|| =", np.linalg.norm(H @ v - hvp(theta0, v)) / np.linalg.norm(H @ v))

# Lanczos driven only by the product v -> hvp(theta0, v): H itself is never passed in.
mL = 20
Qb = np.zeros((nfeat, mL + 1))
al = np.zeros(mL)
be = np.zeros(mL)
w0 = rng.standard_normal(nfeat)
Qb[:, 0] = w0 / np.linalg.norm(w0)
for j in range(mL):
    w = hvp(theta0, Qb[:, j])
    al[j] = Qb[:, j] @ w
    w = w - al[j] * Qb[:, j] - (be[j - 1] * Qb[:, j - 1] if j > 0 else 0.0)
    w = w - Qb[:, :j + 1] @ (Qb[:, :j + 1].T @ w)
    w = w - Qb[:, :j + 1] @ (Qb[:, :j + 1].T @ w)
    be[j] = np.linalg.norm(w)
    Qb[:, j + 1] = w / be[j]
Tl = np.diag(al) + np.diag(be[:-1], 1) + np.diag(be[:-1], -1)
ritz = np.linalg.eigvalsh(Tl)
true = np.linalg.eigvalsh(H)
print(f"Lanczos with {mL} Hessian-vector products on a {nfeat} x {nfeat} Hessian")
print(f"  lambda_max: Ritz {ritz[-1]:.10f}   true {true[-1]:.10f}   error {abs(ritz[-1]-true[-1]):.2e}")
print(f"  lambda_min: Ritz {ritz[0]:.10f}   true {true[0]:.10f}   error {abs(ritz[0]-true[0]):.2e}")
print(f"  stable step size 2 / lambda_max = {2/true[-1]:.6f}")
assert np.linalg.norm(H @ v - hvp(theta0, v)) / np.linalg.norm(H @ v) < 1e-12
assert abs(ritz[-1] - true[-1]) < 1e-6

||H v - hvp(v)|| / ||H v|| = 4.3974019385920333e-16
Lanczos with 20 Hessian-vector products on a 30 x 30 Hessian
  lambda_max: Ritz 90.8097781821   true 90.8097781892   error 7.16e-09
  lambda_min: Ritz -62.0522581023   true -62.0522581024   error 5.30e-11
  stable step size 2 / lambda_max = 0.022024


### Problem L2.3 — Randomized power iteration for a truncated SVD

**Statement.** State the randomized range-finder with $q$ power steps for a rank-$k$ approximation
of $A \in \mathbb{R}^{m \times n}$, explain why the exponent $q$ helps, and measure how the
approximation error approaches the optimum $\sigma_{k+1}$ as $q$ grows.

**Intuition.** Multiplying a Gaussian sketch by $(AA^{\top})^{q}A$ raises every singular value to
the power $2q+1$, which widens the gap between the wanted and unwanted directions before the
sketch is orthogonalized.

**Solution.**

*Step 1 — sketch.* Draw $\Omega \in \mathbb{R}^{n \times (k+p)}$ with independent standard Gaussian
entries and oversampling $p$ (typically $p = 5$).

*Step 2 — power scheme.* Form $Y = (AA^{\top})^{q}A\,\Omega$.

*Step 3 — orthogonalize.* $Y = QR$, so $Q$ has $k+p$ orthonormal columns approximating the
dominant left singular subspace.

*Step 4 — project and factor.* Put $B = Q^{\top}A$, compute its exact SVD
$B = \widetilde{U}\Sigma V^{\top}$, and set $U = Q\widetilde{U}$. Truncating to $k$ columns gives
the approximation.

*Step 5 — why $q$ helps.* The singular values of $(AA^{\top})^{q}A$ are $\sigma_i^{2q+1}$, so the
ratio $\sigma_{k+1}/\sigma_k$ becomes $(\sigma_{k+1}/\sigma_k)^{2q+1}$. Halko, Martinsson and
Tropp's Corollary 10.10 turns this into a spectral-norm bound of the form

$$
\mathbb{E}\,\lVert A - QQ^{\top}A \rVert_2
\;\le\; \left[ \left(1 + \sqrt{\tfrac{k}{p-1}}\right)\sigma_{k+1}^{2q+1}
+ \tfrac{e\sqrt{k+p}}{p}\Bigl( \textstyle\sum_{j \gt k}\sigma_j^{2(2q+1)} \Bigr)^{1/2} \right]^{1/(2q+1)},
$$

whose bracket is raised to the power $1/(2q+1)$: every constant is damped towards $1$ as $q$ grows,
leaving $\sigma_{k+1}$.

$$
\boxed{Y = (AA^{\top})^{q}A\,\Omega, \quad Q = \operatorname{qr}(Y), \quad
\mathbb{E}\lVert A - QQ^{\top}A \rVert_2 \to \sigma_{k+1} \text{ as } q \text{ grows}}
$$

**Key takeaway.** Two power steps are usually enough. The measured errors below are
$1.90\,\sigma_{k+1}$ at $q=0$, $1.003\,\sigma_{k+1}$ at $q=1$ and $1.0009\,\sigma_{k+1}$ at
$q=2$: by $q = 1$ the randomized factorization is already within a third of a percent of the
optimal truncation.

In [24]:
mR, nR, kR, pR = 300, 200, 10, 5
Ur, _ = np.linalg.qr(rng.standard_normal((mR, mR)))
Vr, _ = np.linalg.qr(rng.standard_normal((nR, nR)))
sv = np.array([1.0 / (1 + j) for j in range(nR)])
AR = Ur[:, :nR] @ np.diag(sv) @ Vr.T
print(f"sigma_(k+1) = {sv[kR]:.6f}   (the optimal rank-{kR} spectral-norm error)")
for q in (0, 1, 2, 3):
    Om = rng.standard_normal((nR, kR + pR))
    Y = AR @ Om
    for _ in range(q):
        Y = AR @ (AR.T @ Y)
    Qr, _ = np.linalg.qr(Y)
    Bm = Qr.T @ AR
    Ub, Sb, Vbt = np.linalg.svd(Bm, full_matrices=False)
    Ak = (Qr @ Ub[:, :kR]) @ np.diag(Sb[:kR]) @ Vbt[:kR]
    err = np.linalg.norm(AR - Ak, 2)
    print(f"  q = {q}:  rank-{kR} error {err:.6f}   ratio to sigma_(k+1) {err/sv[kR]:.4f}")
    if q == 2:
        assert err / sv[kR] < 1.01

sigma_(k+1) = 0.090909   (the optimal rank-10 spectral-norm error)
  q = 0:  rank-10 error 0.172820   ratio to sigma_(k+1) 1.9010
  q = 1:  rank-10 error 0.091182   ratio to sigma_(k+1) 1.0030
  q = 2:  rank-10 error 0.090988   ratio to sigma_(k+1) 1.0009
  q = 3:  rank-10 error 0.090909   ratio to sigma_(k+1) 1.0000


### Problem L2.4 — Golub-Kahan bidiagonalization from Lanczos

**Statement.** Derive the Golub-Kahan recurrence by applying Lanczos to the symmetric augmented
matrix $C = \left(\begin{smallmatrix}0 & A \\ A^{\top} & 0\end{smallmatrix}\right)$, define the
matrices $B_k$ and $\widetilde{B}_k$ appearing in
$AV_k = U_kB_k$ and $A^{\top}U_k = V_{k+1}\widetilde{B}_k^{\top}$, and state the flop count of the
dense Householder version.

**Intuition.** $C$ is symmetric, so Lanczos applies; and $C$ maps the top block to the bottom and
back, so a starting vector supported on one block produces iterates that alternate.

**Solution.**

*Step 1 — alternation.* Take $q_1 = (0; v_1)$. Then $Cq_1 = (Av_1; 0)$, supported on the top
block, and applying $C$ again returns to the bottom block. Inductively the Lanczos vectors
alternate between the two blocks.

*Step 2 — the recurrence splits.* Writing the odd vectors as $u_i$ and the even ones as $v_i$, the
three-term Lanczos recurrence of Theorem 4.7 separates into

$$
\alpha_i u_i = A v_i - \beta_{i-1}u_{i-1},
\qquad
\beta_i v_{i+1} = A^{\top}u_i - \alpha_i v_i ,
$$

with $\alpha_i, \beta_i \gt 0$ the normalizing constants. Each half-step touches $A$ or
$A^{\top}$ once.

*Step 3 — the two matrices.* Collect $U_k = [u_1 \cdots u_k]$ and
$V_{k+1} = [v_1 \cdots v_{k+1}]$. Reading the first recurrence column by column gives
$AV_k = U_kB_k$ with $B_k \in \mathbb{R}^{k \times k}$ **upper bidiagonal**, diagonal
$\alpha_1, \dots, \alpha_k$ and superdiagonal $\beta_1, \dots, \beta_{k-1}$.

Reading the second recurrence column by column gives $A^{\top}U_k = V_{k+1}\widetilde{B}_k^{\top}$
with

$$
\widetilde{B}_k = \begin{pmatrix} B_k & \beta_k e_k \end{pmatrix} \in \mathbb{R}^{k \times (k+1)},
$$

that is $B_k$ with one extra **column** $\beta_ke_k$. That extra column is the only difference
between the two identities, and it plays the role of the $\beta_mq_{m+1}e_m^{\top}$ term in
Theorem 4.7.

*Step 4 — singular values.* $B_k$ is $U_k^{\top}AV_k$, so its singular values are Ritz-like
approximations to those of $A$; in the dense Householder version run to completion,
$U^{\top}AV = B$ exactly and $\sigma_i(A) = \sigma_i(B)$.

*Step 5 — cost.* The dense version applies $n$ left and $n-2$ right Householder reflectors, each
$\approx 2mn$ or $2n^{2}$ flops when accumulated over the trailing block, giving

$$
4mn^{2} - \tfrac43 n^{3} \ \text{ flops for } m \ge n .
$$

$$
\boxed{A V_k = U_k B_k, \qquad A^{\top}U_k = V_{k+1}\widetilde{B}_k^{\top},
\qquad \widetilde{B}_k = \begin{pmatrix} B_k & \beta_k e_k \end{pmatrix}}
$$

**Key takeaway.** Nothing ever forms $A^{\top}A$, whose condition number is $\kappa_2(A)^{2}$.
Bidiagonalization is the phase-one step of every dense SVD routine, `numpy.linalg.svd` included.
Stopped early at $k \lt n$ it is a Krylov method and gives the *extreme* singular values first;
run to $k = n$ it is exact.

In [25]:
mB, nB, kB = 40, 12, 8
AB = rng.standard_normal((mB, nB))
v0 = rng.standard_normal(nB)
v0 /= np.linalg.norm(v0)
U = np.zeros((mB, kB))
V = np.zeros((nB, kB + 1))
alpha = np.zeros(kB)
beta = np.zeros(kB)
V[:, 0] = v0
for i in range(kB):
    w = AB @ V[:, i] - (beta[i - 1] * U[:, i - 1] if i > 0 else 0.0)
    w -= U[:, :i] @ (U[:, :i].T @ w)                 # reorthogonalize for stability
    w -= U[:, :i] @ (U[:, :i].T @ w)
    alpha[i] = np.linalg.norm(w)
    U[:, i] = w / alpha[i]
    z = AB.T @ U[:, i] - alpha[i] * V[:, i]
    z -= V[:, :i + 1] @ (V[:, :i + 1].T @ z)
    z -= V[:, :i + 1] @ (V[:, :i + 1].T @ z)
    beta[i] = np.linalg.norm(z)
    V[:, i + 1] = z / beta[i]
Bk = np.diag(alpha) + np.diag(beta[:-1], 1)
Btil = np.hstack([Bk, beta[kB - 1] * np.eye(kB)[:, [kB - 1]]])
print("B_k shape:", Bk.shape, "   B_k tilde shape:", Btil.shape)
print("||A V_k - U_k B_k||_F          =", np.linalg.norm(AB @ V[:, :kB] - U @ Bk))
print("||A^T U_k - V_(k+1) Btilde^T||_F =",
      np.linalg.norm(AB.T @ U - V[:, :kB + 1] @ Btil.T))
print("orthonormality: ||U^T U - I|| =", np.abs(U.T @ U - np.eye(kB)).max(),
      "  ||V^T V - I|| =", np.abs(V.T @ V - np.eye(kB + 1)).max())
sv_true = np.linalg.svd(AB, compute_uv=False)
sv_bidi = np.linalg.svd(Bk, compute_uv=False)
print("top 3 singular values of A  :", sv_true[:3])
print(f"top 3 from B_k after k = {kB} :", sv_bidi[:3])
print("relative errors             :", np.abs(sv_bidi[:3] - sv_true[:3]) / sv_true[:3])
assert np.linalg.norm(AB @ V[:, :kB] - U @ Bk) < 1e-10
assert np.linalg.norm(AB.T @ U - V[:, :kB + 1] @ Btil.T) < 1e-10
assert abs(sv_bidi[0] - sv_true[0]) / sv_true[0] < 1e-4

# Run to completion (k = n) and the singular values become exact.
kfull = nB
Uf = np.zeros((mB, kfull))
Vf = np.zeros((nB, kfull + 1))
al_f = np.zeros(kfull)
be_f = np.zeros(kfull)
Vf[:, 0] = v0
for i in range(kfull):
    w = AB @ Vf[:, i] - (be_f[i - 1] * Uf[:, i - 1] if i > 0 else 0.0)
    w -= Uf[:, :i] @ (Uf[:, :i].T @ w)
    w -= Uf[:, :i] @ (Uf[:, :i].T @ w)
    al_f[i] = np.linalg.norm(w)
    Uf[:, i] = w / al_f[i]
    z = AB.T @ Uf[:, i] - al_f[i] * Vf[:, i]
    z -= Vf[:, :i + 1] @ (Vf[:, :i + 1].T @ z)
    z -= Vf[:, :i + 1] @ (Vf[:, :i + 1].T @ z)
    be_f[i] = np.linalg.norm(z)
    Vf[:, i + 1] = z / be_f[i] if be_f[i] > 1e-14 else 0.0
Bfull = np.diag(al_f) + np.diag(be_f[:-1], 1)
sv_full = np.linalg.svd(Bfull, compute_uv=False)
print(f"complete bidiagonalization k = {kfull}: max |sigma(B) - sigma(A)| ="
      f" {np.abs(sv_full - sv_true).max():.3e}")
assert np.abs(sv_full - sv_true).max() < 1e-12

B_k shape: (8, 8)    B_k tilde shape: (8, 9)
||A V_k - U_k B_k||_F          = 4.391969388385915e-15
||A^T U_k - V_(k+1) Btilde^T||_F = 5.108776835332328e-15
orthonormality: ||U^T U - I|| = 3.3306690738754696e-16   ||V^T V - I|| = 2.220446049250313e-16
top 3 singular values of A  : [9.8719 8.9777 8.2401]
top 3 from B_k after k = 8 : [9.8719 8.9758 8.2127]
relative errors             : [0.     0.0002 0.0033]
complete bidiagonalization k = 12: max |sigma(B) - sigma(A)| = 5.329e-15


### Problem L2.5 — Singular values as eigenvalues of an augmented matrix

**Statement.** For $A \in \mathbb{R}^{m \times n}$ prove that the non-zero singular values of $A$
are exactly the positive eigenvalues of
$C = \left(\begin{smallmatrix}0 & A \\ A^{\top} & 0\end{smallmatrix}\right)$, and that they occur
in $\pm$ pairs.

**Intuition.** A singular triplet couples $u$ and $v$ through $A$ and $A^{\top}$; stacking them
turns the coupling into a single eigenvalue equation.

**Solution.**

*Step 1.* Let $(\sigma, u, v)$ satisfy $Av = \sigma u$, $A^{\top}u = \sigma v$ with
$\lVert u \rVert = \lVert v \rVert = 1$ and $\sigma \gt 0$.

*Step 2.* With $x_{+} = (u; v)$,

$$
C x_{+} = \begin{pmatrix} Av \\ A^{\top}u \end{pmatrix} = \begin{pmatrix} \sigma u \\ \sigma v \end{pmatrix} = \sigma x_{+} .
$$

*Step 3.* With $x_{-} = (u; -v)$,

$$
C x_{-} = \begin{pmatrix} -Av \\ A^{\top}u \end{pmatrix} = \begin{pmatrix} -\sigma u \\ \sigma v \end{pmatrix} = -\sigma x_{-} .
$$

*Step 4.* $x_{+}$ and $x_{-}$ are orthogonal and non-zero, so each $\sigma$ contributes the pair
$\pm\sigma$. Counting: $A$ has $r = \operatorname{rank}(A)$ non-zero singular values, giving $2r$
eigenvalues, and $C$ has $\operatorname{rank}(C) = 2r$, so the remaining $m+n-2r$ eigenvalues are
zero and nothing is missed.

$$
\boxed{\operatorname{spec}(C) = \lbrace \pm\sigma_1, \dots, \pm\sigma_r \rbrace \cup \lbrace 0 \rbrace^{m+n-2r}}
$$

**Key takeaway.** Any symmetric eigensolver becomes an SVD solver. The price is that the
eigenvalues of $C$ come in $\pm$ pairs of equal modulus, which is exactly the situation the
unshifted QR algorithm cannot handle (Problem L1.4) — hence the specialized bidiagonal SVD step of
Problem L2.4.

In [26]:
mC, nC = 6, 4
AC = rng.standard_normal((mC, nC))
C = np.block([[np.zeros((mC, mC)), AC], [AC.T, np.zeros((nC, nC))]])
evC = np.linalg.eigvalsh(C)
svA = np.linalg.svd(AC, compute_uv=False)
print("eigenvalues of C :", np.round(evC, 6))
print("singular values  :", np.round(svA, 6))
print("positive eigenvalues of C:", np.round(evC[evC > 1e-10], 6))
assert np.allclose(np.sort(evC[evC > 1e-10]), np.sort(svA))
assert np.allclose(np.sort(-evC[evC < -1e-10]), np.sort(svA))

eigenvalues of C : [-3.9691 -2.3949 -1.1752 -0.9171 -0.      0.      0.9171  1.1752  2.3949
  3.9691]
singular values  : [3.9691 2.3949 1.1752 0.9171]
positive eigenvalues of C: [0.9171 1.1752 2.3949 3.9691]


### Problem L2.6 — Bauer-Fike and why normal matrices are safe

**Statement.** State the Bauer-Fike theorem for a diagonalizable $A = V\Lambda V^{-1}$ perturbed
by $E$, and evaluate the bound when $A$ is normal.

**Intuition.** Perturbing eigenvalues costs the condition number of the eigenvector basis; an
orthonormal basis costs nothing.

**Solution.**

*Step 1 — statement.* If $\mu \in \operatorname{spec}(A+E)$ then there is
$\lambda \in \operatorname{spec}(A)$ with

$$
\lvert \mu - \lambda \rvert \;\le\; \kappa_2(V)\,\lVert E \rVert_{\mathrm{op}},
\qquad \kappa_2(V) = \lVert V \rVert_{\mathrm{op}}\lVert V^{-1} \rVert_{\mathrm{op}} .
$$

*Step 2 — normal case.* If $A^{\ast}A = AA^{\ast}$ then $A = Q\Lambda Q^{\ast}$ with $Q$ unitary,
so $V$ may be taken unitary.

*Step 3.* For a unitary $Q$, $\lVert Q \rVert_{\mathrm{op}} = \lVert Q^{\ast} \rVert_{\mathrm{op}} = 1$,
hence $\kappa_2(Q) = 1$.

$$
\boxed{\lvert \mu - \lambda \rvert \le \kappa_2(V)\lVert E \rVert_{\mathrm{op}};
\quad \text{normal } A: \ \lvert \mu - \lambda \rvert \le \lVert E \rVert_{\mathrm{op}}}
$$

**Key takeaway.** Every algorithm in this module is backward stable — it returns the exact
spectrum of $A + E$ with $\lVert E \rVert = O(u\lVert A \rVert)$ — so Bauer-Fike converts backward
stability into forward accuracy, and the conversion factor is $\kappa_2(V)$. For symmetric input
that factor is $1$, which is why symmetric eigensolvers are trusted absolutely and non-symmetric
ones are not.

In [27]:
Msym = rng.standard_normal((6, 6))
Anorm = (Msym + Msym.T) / 2.0
Vd = np.array([[1.0, 1.0], [0.0, 1e-6]])           # nearly singular eigenvector matrix
Anon = Vd @ np.diag([1.0, 2.0]) @ np.linalg.inv(Vd)
for name, A, V in (("symmetric (normal)", Anorm, np.linalg.eigh(Anorm)[1]),
                   ("ill-conditioned basis", Anon, Vd)):
    E = 1e-8 * rng.standard_normal(A.shape)
    E = (E + E.T) / 2.0 if name.startswith("symmetric") else E
    lam0 = np.sort(np.linalg.eigvals(A).real)
    lam1 = np.sort(np.linalg.eigvals(A + E).real)
    kappa = np.linalg.cond(V)
    print(f"  {name:22s}  kappa_2(V) = {kappa:.3e}   ||E|| = {np.linalg.norm(E,2):.2e}"
          f"   max |d lambda| = {np.abs(lam1-lam0).max():.3e}"
          f"   bound = {kappa*np.linalg.norm(E,2):.3e}")
    assert np.abs(lam1 - lam0).max() <= kappa * np.linalg.norm(E, 2) + 1e-12

  symmetric (normal)      kappa_2(V) = 1.000e+00   ||E|| = 3.44e-08   max |d lambda| = 2.459e-08   bound = 3.444e-08
  ill-conditioned basis   kappa_2(V) = 2.000e+06   ||E|| = 1.67e-08   max |d lambda| = 6.972e-03   bound = 3.331e-02


### Problem L2.7 — Operation count of Lanczos

**Statement.** Count the flops for $m$ steps of Lanczos on a symmetric sparse
$A \in \mathbb{R}^{n \times n}$ with $\operatorname{nnz}(A) = N$, without reorthogonalization.

**Intuition.** Each step is one sparse matrix-vector product plus a fixed number of length-$n$
vector operations.

**Solution.**

*Step 1.* Sparse product $w = Aq_j$: one multiply and one add per non-zero, $2N$ flops.

*Step 2.* Inner product $\alpha_j = q_j^{\top}w$: $2n$ flops.

*Step 3.* Update $w \gets w - \alpha_j q_j - \beta_{j-1}q_{j-1}$: two scaled subtractions, $4n$
flops.

*Step 4.* Norm $\beta_j = \lVert w \rVert$: $2n$ flops.

*Step 5.* Normalize $q_{j+1} = w/\beta_j$: $n$ flops.

*Step 6.* Total per step $2N + 9n$; over $m$ steps,

$$
m\,(2N + 9n) .
$$

$$
\boxed{m\bigl(2\operatorname{nnz}(A) + 9n\bigr) \text{ flops, } O(mn) \text{ memory with three vectors}}
$$

**Key takeaway.** Linear in $N$ and in $m$: this is why Lanczos scales to $n = 10^{6}$. Full
reorthogonalization adds $4mn$ per step, i.e. $2m^{2}n$ overall, which is what buys the flat
orthogonality curve of Section 7.5 — and what eventually dominates the cost.

In [28]:
def lanczos_flops(m, N, n, reorth=False):
    per_step = 2 * N + 9 * n
    return m * per_step + (2 * m ** 2 * n if reorth else 0)


for n_, N_, m_ in ((10 ** 6, 7 * 10 ** 6, 100), (10 ** 4, 5 * 10 ** 4, 200)):
    plain = lanczos_flops(m_, N_, n_)
    full = lanczos_flops(m_, N_, n_, reorth=True)
    dense = 4.0 / 3.0 * n_ ** 3
    print(f"  n = {n_:.0e}, nnz = {N_:.0e}, m = {m_}:  Lanczos {plain:.3e} flops,"
          f" with reorth {full:.3e}, dense tridiagonalization {dense:.3e}")
    assert plain < dense

  n = 1e+06, nnz = 7e+06, m = 100:  Lanczos 2.300e+09 flops, with reorth 2.230e+10, dense tridiagonalization 1.333e+18
  n = 1e+04, nnz = 5e+04, m = 200:  Lanczos 3.800e+07 flops, with reorth 8.380e+08, dense tridiagonalization 1.333e+12


### Problem L2.8 — Physics: normal modes of a three-mass spring chain

**Statement.** Three equal masses $m$ lie on a line, joined to each other and to two fixed walls by
four identical springs of stiffness $\kappa$. Find the three natural angular frequencies, identify
the mode shapes, and give the rate at which shifted inverse iteration with $\mu = 0$ finds the
lowest one.

**Intuition.** Small oscillations of a linear chain are an eigenvalue problem for the stiffness
matrix, and the modes are the eigenvectors.

**Solution.**

*Step 1 — equations of motion.* Newton's law for the displacements $x = (x_1,x_2,x_3)^{\top}$ is
$m\ddot{x} = -\kappa K x$ with

$$
K = \begin{pmatrix} 2 & -1 & 0 \\ -1 & 2 & -1 \\ 0 & -1 & 2 \end{pmatrix} .
$$

*Step 2 — separate.* Substituting $x = v\,e^{i\omega t}$ gives $Kv = (m\omega^{2}/\kappa)\,v$, so
$\omega_i^{2} = (\kappa/m)\,\lambda_i(K)$.

*Step 3 — spectrum.* $K$ is the running matrix $T$ of Section 6 after flipping the sign of the
middle coordinate, a diagonal similarity, so it has the same spectrum
$\lbrace 2+\sqrt2,\ 2,\ 2-\sqrt2 \rbrace$.

*Step 4 — frequencies.*

$$
\omega_1 = \sqrt{(2-\sqrt2)\tfrac{\kappa}{m}} \approx 0.7654\sqrt{\tfrac{\kappa}{m}},
\quad
\omega_2 = \sqrt{2\tfrac{\kappa}{m}} \approx 1.4142\sqrt{\tfrac{\kappa}{m}},
\quad
\omega_3 = \sqrt{(2+\sqrt2)\tfrac{\kappa}{m}} \approx 1.8478\sqrt{\tfrac{\kappa}{m}} .
$$

*Step 5 — mode shapes.* The eigenvectors of $K$ are $(1,\sqrt2,1)$ (all masses in phase, the
"breathing" mode), $(1,0,-1)$ (centre mass at rest), and $(1,-\sqrt2,1)$ (neighbours in
antiphase).

*Step 6 — rate.* The lowest frequency is the smallest eigenvalue, so Theorem 4.1 with $\mu = 0$
gives $r = (2-\sqrt2)/2 \approx 0.2929$ per step — the rate measured in Example 6.1.

$$
\boxed{\omega^{2} \in \tfrac{\kappa}{m}\lbrace 2-\sqrt2,\ 2,\ 2+\sqrt2 \rbrace,
\qquad r = 1 - \tfrac{\sqrt2}{2} \approx 0.2929}
$$

**Key takeaway.** Engineering cares about the **lowest** mode — the one a bridge or a wing is
excited into — and that is the eigenvalue power iteration converges to last and inverse iteration
converges to first.

In [29]:
kappa, mass = 1.0, 1.0
K = np.array([[2.0, -1.0, 0.0], [-1.0, 2.0, -1.0], [0.0, -1.0, 2.0]])
w2 = np.linalg.eigvalsh(kappa / mass * K)
print("omega^2 :", w2, "  exact:", [2 - np.sqrt(2), 2.0, 2 + np.sqrt(2)])
print("omega   :", np.sqrt(w2))
for i, v in enumerate(np.linalg.eigh(K)[1].T):
    print(f"  mode {i+1}: shape {np.round(v/np.abs(v).max(), 4)}")
x = rng.standard_normal(3)
x /= np.linalg.norm(x)
q_low = np.linalg.eigh(K)[1][:, 0]
sins = []
for _ in range(15):
    sins.append(np.linalg.norm(x - (q_low @ x) * q_low))
    y = np.linalg.solve(K, x)
    x = y / np.linalg.norm(y)
obs = np.exp(np.polyfit(np.arange(3, 12), np.log(np.array(sins)[3:12]), 1)[0])
print(f"inverse iteration: observed rate {obs:.6f}   predicted {(2-np.sqrt(2))/2:.6f}")
print(f"lowest frequency  : {np.sqrt(x @ K @ x):.10f}   exact {np.sqrt(2-np.sqrt(2)):.10f}")
assert np.allclose(w2, [2 - np.sqrt(2), 2.0, 2 + np.sqrt(2)])
assert abs(obs - (2 - np.sqrt(2)) / 2) < 5e-3

omega^2 : [0.5858 2.     3.4142]   exact: [np.float64(0.5857864376269049), 2.0, np.float64(3.414213562373095)]
omega   : [0.7654 1.4142 1.8478]
  mode 1: shape [-0.7071 -1.     -0.7071]
  mode 2: shape [-1.  0.  1.]
  mode 3: shape [ 0.7071 -1.      0.7071]
inverse iteration: observed rate 0.290827   predicted 0.292893
lowest frequency  : 0.7653668647   exact 0.7653668647


### Problem L2.9 — Physics: ground-state energy of a quantum harmonic oscillator

**Statement.** In units $\hbar = m = \omega = 1$ the Hamiltonian is
$H = -\tfrac12 \tfrac{d^{2}}{dx^{2}} + \tfrac12 x^{2}$, with exact energy levels
$E_n = n + \tfrac12$. Discretize on $[-5,5]$ with $N$ interior points and compute the four lowest
levels by Lanczos. Explain why the ground state is *harder* for Lanczos than the top of the
spectrum, and what to do about it.

**Intuition.** Krylov methods find the extremes of the spectrum. The ground state is the smallest
eigenvalue, and after discretization the top of the spectrum is $O(h^{-2})$ while the bottom is
$O(1)$ — so the bottom is buried in a dense cluster.

**Solution.**

*Step 1 — discretize.* With spacing $h$ and the second-difference stencil,

$$
H \approx \frac{1}{2h^{2}}\operatorname{tridiag}(-1, 2, -1) + \tfrac12\operatorname{diag}(x_i^{2}) .
$$

*Step 2 — the spectrum is badly graded.* The kinetic term has eigenvalues up to $\approx 2/h^{2}$,
so at $N = 400$ on $[-5,5]$ the largest eigenvalue is $3223.4$ while the wanted ones sit near
$0.5, 1.5, 2.5, 3.5$.

*Step 3 — why plain Lanczos fails.* Running Lanczos on $-H$ makes the ground state the largest
eigenvalue, but Krylov convergence is governed by the **relative** gap. Here

$$
\frac{E_1 - E_0}{E_{\max} - E_0} = \frac{1}{3222.9} = 3.1 \times 10^{-4} ,
$$

so the ground state is buried. Sixty steps return $3.12$ instead of $0.5$ — wrong by a factor of
six, and wrong in a way no residual test on the *tridiagonal* problem would reveal.

*Step 4 — the repair is a shift and an inverse.* Apply Lanczos to $H^{-1}$ instead. Its spectrum is
$\lbrace 1/E_n \rbrace$, whose largest values are $2$, $0.667$, $0.400$, $\dots$, so the relative
gap becomes

$$
\frac{1/E_0 - 1/E_1}{1/E_0 - 1/E_{\max}} = 0.667 ,
$$

three and a half orders of magnitude better. Thirty steps then give all four levels to $10^{-13}$.
This is Theorem 4.1 used as a preconditioner: inverting turns an interior or badly separated
eigenvalue into a well-separated extreme one.

*Step 5 — accuracy.* The remaining error against the exact $n + \tfrac12$ is $O(h^{2})$
discretization error, about $2 \times 10^{-5}$ at $N = 400$. Refining the grid, not iterating
longer, is what improves it.

$$
\boxed{E_n \approx n + \tfrac12; \quad \text{plain Lanczos on } -H \text{ fails, Lanczos on } H^{-1} \text{ converges}}
$$

**Key takeaway.** Two different errors live here: the iteration error, which Lanczos drives to
machine precision in tens of steps, and the discretization error, which it cannot touch. Reporting
one as the other is the classic mistake in computational physics.

In [30]:
L, Nq = 10.0, 400
xs = np.linspace(-L / 2, L / 2, Nq + 2)[1:-1]
hq = xs[1] - xs[0]
Hq = (np.diag(2.0 * np.ones(Nq)) + np.diag(-np.ones(Nq - 1), 1)
      + np.diag(-np.ones(Nq - 1), -1)) / (2.0 * hq ** 2) + np.diag(0.5 * xs ** 2)
dense_levels = np.linalg.eigvalsh(Hq)
print("dense eigensolver, four lowest levels:", dense_levels[:4], "  exact 0.5, 1.5, 2.5, 3.5")
print(f"largest eigenvalue of the discretization: {dense_levels[-1]:.1f}   (order 1/h^2)")
print(f"relative gap at the bottom of H: "
      f"{(dense_levels[1]-dense_levels[0])/(dense_levels[-1]-dense_levels[0]):.3e}")

a_q, b_q, _ = lanczos(-Hq, rng.standard_normal(Nq), 60, reorth=True)
Tq = np.diag(a_q) + np.diag(b_q[:-1], 1) + np.diag(b_q[:-1], -1)
plain = -np.linalg.eigvalsh(Tq)[::-1]
print("\nplain Lanczos on -H, 60 steps, four lowest:", plain[:4])
print("  error against the dense solver          :", np.abs(plain[:4] - dense_levels[:4]))

Hinv = np.linalg.inv(Hq)
Hinv = (Hinv + Hinv.T) / 2.0
print(f"relative gap at the top of H^-1: "
      f"{(1/dense_levels[0]-1/dense_levels[1])/(1/dense_levels[0]-1/dense_levels[-1]):.3f}")
a_i, b_i, _ = lanczos(Hinv, rng.standard_normal(Nq), 30, reorth=True)
Ti = np.diag(a_i) + np.diag(b_i[:-1], 1) + np.diag(b_i[:-1], -1)
shifted = 1.0 / np.linalg.eigvalsh(Ti)[::-1]
print("shift-and-invert Lanczos on H^-1, 30 steps, four lowest:", shifted[:4])
print("  iteration error against the dense solver :", np.abs(shifted[:4] - dense_levels[:4]))
print(f"  discretization error of the ground state : {abs(dense_levels[0]-0.5):.2e}")
assert abs(plain[0] - dense_levels[0]) > 0.1
assert np.abs(shifted[:4] - dense_levels[:4]).max() < 1e-10
assert abs(dense_levels[0] - 0.5) < 1e-4

dense eigensolver, four lowest levels:

 [0.5    1.4999 2.4997 3.4995]   exact 0.5, 1.5, 2.5, 3.5
largest eigenvalue of the discretization: 3223.4   (order 1/h^2)
relative gap at the bottom of H: 3.103e-04



plain Lanczos on -H, 60 steps, four lowest: [ 3.1165  7.94   16.9566 28.7992]
  error against the dense solver          : [ 2.6165  6.4401 14.4569 25.2997]


relative gap at the top of H^-1: 0.667


shift-and-invert Lanczos on H^-1, 30 steps, four lowest: [0.5    1.4999 2.4997 3.4995]
  iteration error against the dense solver : [0. 0. 0. 0.]
  discretization error of the ground state : 1.94e-05


### Problem L2.10 — The spectral norm of a weight matrix by power iteration

**Statement.** Spectral normalization replaces a layer weight $W$ by $W/\sigma_1(W)$ so that the
layer is $1$-Lipschitz. Show that one power-iteration step on $W^{\top}W$ can be written as two
matrix-vector products with $W$ and $W^{\top}$, give the convergence rate, and check the estimate
against `numpy.linalg.norm(W, 2)`.

**Intuition.** $\sigma_1(W)^{2} = \lambda_1(W^{\top}W)$, and $W^{\top}W$ need never be formed.

**Solution.**

*Step 1 — the eigenvalue problem.* $\sigma_1(W)$ is the square root of the largest eigenvalue of
$W^{\top}W$, which is symmetric positive semidefinite.

*Step 2 — the product without the matrix.* $(W^{\top}W)v = W^{\top}(Wv)$: one product with $W$,
one with $W^{\top}$, at $O(\operatorname{nnz})$ each. Forming $W^{\top}W$ would cost $O(mn^{2})$
and square the condition number.

*Step 3 — the iteration.* $u \gets Wv/\lVert Wv \rVert$, then $v \gets W^{\top}u/\lVert W^{\top}u \rVert$;
the estimate is $\sigma \approx u^{\top}Wv$.

*Step 4 — two rates, not one.* Power iteration on $W^{\top}W$ contracts the *direction* error at
$\lambda_2/\lambda_1 = (\sigma_2/\sigma_1)^{2}$ per step. The estimate $u^{\top}Wv$ is a Rayleigh
quotient of $W^{\top}W$ up to a square root, and a Rayleigh quotient is second-order accurate
([Module 06](../06_eigenvalues_eigenvectors_spectral_theory/first_principles.ipynb), Theorem 4.7),
so the *value* error contracts at

$$
\left( \frac{\sigma_2}{\sigma_1} \right)^{4} \text{ per step.}
$$

$$
\boxed{\sigma_1(W) \approx u^{\top}Wv; \quad \text{direction rate } (\sigma_2/\sigma_1)^{2},
\quad \text{value rate } (\sigma_2/\sigma_1)^{4}}
$$

**Key takeaway.** In training, one such step is taken per minibatch and the vectors are carried
over, so the estimate tracks a slowly changing $W$ at the cost of two matrix-vector products —
power iteration amortized across an optimization run. The rate depends entirely on the gap: a
freshly initialized Gaussian layer has $\sigma_2/\sigma_1 \approx 0.99$ and one step per
minibatch is barely enough, while a trained layer with a dominant direction converges in a handful
of steps.

In [31]:
mW, nW = 64, 48
W0 = rng.standard_normal((mW, nW)) / np.sqrt(nW)
uplant = rng.standard_normal(mW)
uplant /= np.linalg.norm(uplant)
vplant = rng.standard_normal(nW)
vplant /= np.linalg.norm(vplant)
W = W0 + 2.5 * np.outer(uplant, vplant)          # a planted dominant direction
Ufull, sv, Vt = np.linalg.svd(W, full_matrices=False)
v1 = Vt[0]
print(f"sigma_1 = {sv[0]:.10f}   sigma_2 = {sv[1]:.10f}   sigma_2/sigma_1 = {sv[1]/sv[0]:.6f}")

v = rng.standard_normal(nW)
v /= np.linalg.norm(v)
sins, ests = [], []
for k in range(30):
    u = W @ v
    u /= np.linalg.norm(u)
    v = W.T @ u
    v /= np.linalg.norm(v)
    sins.append(np.linalg.norm(v - (v1 @ v) * v1))
    ests.append(u @ W @ v)
sins = np.array(sins)
err = np.abs(np.array(ests) - sv[0])
fit = np.arange(3, 14)
rate_dir = np.exp(np.polyfit(fit, np.log(sins[3:14]), 1)[0])
rate_val = np.exp(np.polyfit(fit, np.log(err[3:14]), 1)[0])
print(f"numpy spectral norm : {np.linalg.norm(W, 2):.12f}")
print(f"power iteration     : {ests[-1]:.12f}   error {err[-1]:.2e}")
print(f"direction rate observed {rate_dir:.6f}   predicted (s2/s1)^2 = {(sv[1]/sv[0])**2:.6f}")
print(f"value     rate observed {rate_val:.6f}   predicted (s2/s1)^4 = {(sv[1]/sv[0])**4:.6f}")
print(f"Lipschitz constant of W / sigma_1: {np.linalg.norm(W/ests[-1], 2):.12f}")
assert abs(ests[-1] - np.linalg.norm(W, 2)) < 1e-10
assert abs(rate_dir - (sv[1] / sv[0]) ** 2) < 0.02
assert abs(rate_val - (sv[1] / sv[0]) ** 4) < 0.02

sigma_1 = 2.9159525212   sigma_2 = 2.0495488791   sigma_2/sigma_1 = 0.702875
numpy spectral norm : 2.915952521167
power iteration     : 2.915952521167   error 2.22e-15
direction rate observed 0.479596   predicted (s2/s1)^2 = 0.494033
value     rate observed 0.227525   predicted (s2/s1)^4 = 0.244068
Lipschitz constant of W / sigma_1: 1.000000000000


## L3 — Challenge Proofs

### Problem L3.1 — Courant-Fischer, and the interlacing it forces on Ritz values

**Statement.** For symmetric $A$ with $\lambda_1 \ge \dots \ge \lambda_n$, Courant-Fischer says

$$
\lambda_k = \max_{\dim S = k}\ \min_{0 \neq x \in S} R_A(x) .
$$

Take this as given — it is proved in
[Module 06](../06_eigenvalues_eigenvectors_spectral_theory/first_principles.ipynb) as Proof 5.4,
in this same descending convention. Use it to prove that for any $Q \in \mathbb{R}^{n \times m}$
with orthonormal columns, the Ritz values $\theta_1 \ge \dots \ge \theta_m$ of
$H = Q^{\top}AQ$ satisfy

$$
\lambda_i(A) \;\ge\; \theta_i \;\ge\; \lambda_{n-m+i}(A), \qquad i = 1, \dots, m .
$$

**Intuition.** Ritz values are eigenvalues of $A$ restricted to a subspace, so they optimize over
fewer subspaces than the true eigenvalues do — the max-min can only shrink.

**Solution.**

*Step 1 — the Rayleigh quotients agree.* For $z \in \mathbb{R}^{m}$, $z \neq 0$, orthonormality of
$Q$ gives $\lVert Qz \rVert = \lVert z \rVert$, so

$$
R_H(z) = \frac{z^{\top}Q^{\top}AQz}{z^{\top}z} = \frac{(Qz)^{\top}A(Qz)}{(Qz)^{\top}(Qz)} = R_A(Qz) .
$$

*Step 2 — upper bound.* Apply Courant-Fischer to $H$:

$$
\theta_i = \max_{S \subseteq \mathbb{R}^{m},\ \dim S = i}\ \min_{0 \neq z \in S} R_A(Qz)
= \max_{\substack{U \subseteq \operatorname{Col}(Q) \\ \dim U = i}}\ \min_{0 \neq x \in U} R_A(x) ,
$$

because $S \mapsto QS$ is a bijection between $i$-dimensional subspaces of $\mathbb{R}^{m}$ and
$i$-dimensional subspaces of $\operatorname{Col}(Q)$. The right side maximizes over a
**subfamily** of all $i$-dimensional subspaces of $\mathbb{R}^{n}$, so

$$
\theta_i \;\le\; \max_{\dim U = i}\ \min_{0 \neq x \in U} R_A(x) = \lambda_i(A) .
$$

*Step 3 — lower bound by reflection.* Apply Step 2 to $-A$, whose Ritz matrix is $-H$. Eigenvalues
reverse and reindex: $\lambda_j(-A) = -\lambda_{n-j+1}(A)$ and
$\theta_j(-H) = -\theta_{m-j+1}(H)$. Step 2 for $-A$ reads
$-\theta_{m-j+1} \le -\lambda_{n-j+1}$, that is
$\theta_{m-j+1} \ge \lambda_{n-j+1}$.

*Step 4 — reindex.* Put $i = m-j+1$, so $j = m-i+1$ and $n-j+1 = n-m+i$:

$$
\theta_i \;\ge\; \lambda_{n-m+i}(A) .
$$

$$
\boxed{\lambda_i(A) \ \ge\ \theta_i \ \ge\ \lambda_{n-m+i}(A)}
$$

**Key takeaway.** Ritz values are always **inside** the spectrum and always pessimistic at both
ends. At $m = n$ the two bounds coincide and the Ritz values are the eigenvalues; at $m \ll n$ the
gap explains why Krylov methods approach $\lambda_1$ from below and $\lambda_n$ from above, never
overshooting.

In [32]:
Ms = rng.standard_normal((10, 10))
As = (Ms + Ms.T) / 2.0
lamA = np.linalg.eigvalsh(As)[::-1]
for m in (3, 6, 9):
    Qm, _ = np.linalg.qr(rng.standard_normal((10, m)))
    th = np.linalg.eigvalsh(Qm.T @ As @ Qm)[::-1]
    ok = all(lamA[i] >= th[i] - 1e-12 >= lamA[10 - m + i] - 1e-12 for i in range(m))
    print(f"  m = {m}: Ritz {np.round(th, 4)}   interlacing holds: {ok}")
    assert ok

  m = 3: Ritz [ 1.3689 -0.5609 -1.1056]   interlacing holds: True
  m = 6: Ritz [ 3.1735  1.9383  1.0486  0.2219 -0.9614 -2.2144]   interlacing holds: True
  m = 9: Ritz [ 4.0058  3.1922  2.0286  0.7016  0.3782 -0.7887 -1.2694 -1.8367 -3.4804]   interlacing holds: True


### Problem L3.2 — The Hoffman-Wielandt inequality

**Statement.** Let $A, B \in \mathbb{R}^{n \times n}$ be symmetric with eigenvalues
$\lambda_1 \ge \dots \ge \lambda_n$ and $\mu_1 \ge \dots \ge \mu_n$. Prove

$$
\sum_{i=1}^{n} (\mu_i - \lambda_i)^{2} \;\le\; \lVert A - B \rVert_F^{2} .
$$

**Intuition.** Expand the Frobenius norm in the two eigenbases; what is left is a bilinear form in
a doubly stochastic matrix, and its extreme value is attained at a permutation.

**Solution.**

*Step 1 — diagonalize both.* Write $A = Q\Lambda Q^{\top}$ and $B = U\!M\,U^{\top}$ with $Q, U$
orthogonal, $\Lambda = \operatorname{diag}(\lambda)$, $M = \operatorname{diag}(\mu)$. Set
$W = Q^{\top}U$, again orthogonal. Frobenius norm is orthogonally invariant, so

$$
\lVert A - B \rVert_F^{2} = \lVert \Lambda - W M W^{\top} \rVert_F^{2} .
$$

*Step 2 — expand.*

$$
\lVert \Lambda - WMW^{\top} \rVert_F^{2}
= \sum_i \lambda_i^{2} + \sum_i \mu_i^{2} - 2\operatorname{tr}\bigl( \Lambda W M W^{\top} \bigr),
$$

and $\operatorname{tr}(\Lambda W M W^{\top}) = \sum_{i,j} \lambda_i \mu_j W_{ij}^{2}$.

*Step 3 — the matrix of squares is doubly stochastic.* Put $S_{ij} = W_{ij}^{2}$. Since $W$ is
orthogonal, every row and every column of $S$ sums to $1$, and $S_{ij} \ge 0$.

*Step 4 — maximize the bilinear form.* By Birkhoff's theorem the doubly stochastic matrices are
the convex hull of the permutation matrices, and $S \mapsto \sum_{i,j}\lambda_i\mu_j S_{ij}$ is
linear, so its maximum over that set is attained at a permutation $P_\pi$, where it equals
$\sum_i \lambda_i \mu_{\pi(i)}$.

*Step 5 — the best permutation is the identity.* By the rearrangement inequality, for two
sequences both sorted in decreasing order,
$\sum_i \lambda_i\mu_{\pi(i)} \le \sum_i \lambda_i\mu_i$ for every permutation $\pi$.

*Step 6 — assemble.*

$$
\lVert A - B \rVert_F^{2} \;\ge\; \sum_i \lambda_i^{2} + \sum_i \mu_i^{2} - 2\sum_i \lambda_i\mu_i
= \sum_i (\mu_i - \lambda_i)^{2} .
$$

$$
\boxed{\left( \sum_{i=1}^{n} (\mu_i - \lambda_i)^{2} \right)^{1/2} \le \lVert A - B \rVert_F}
$$

**Key takeaway.** Weyl's inequality bounds each eigenvalue separately by the operator norm; this
bounds the whole sorted spectrum at once by the Frobenius norm. It is the tool for statements about
*all* the eigenvalues of a perturbed matrix, such as the accuracy of a whole computed spectrum.

In [33]:
for trial in range(4):
    M1 = rng.standard_normal((7, 7))
    A = (M1 + M1.T) / 2.0
    E = 0.3 * rng.standard_normal((7, 7))
    E = (E + E.T) / 2.0
    lam = np.linalg.eigvalsh(A)[::-1]
    mu = np.linalg.eigvalsh(A + E)[::-1]
    lhs = np.sqrt(np.sum((mu - lam) ** 2))
    rhs = np.linalg.norm(E)
    print(f"  trial {trial}: sqrt(sum of squared shifts) = {lhs:.6f}   ||E||_F = {rhs:.6f}"
          f"   ratio {lhs/rhs:.4f}")
    assert lhs <= rhs + 1e-12

  trial 0: sqrt(sum of squared shifts) = 0.554745   ||E||_F = 1.289824   ratio 0.4301
  trial 1: sqrt(sum of squared shifts) = 0.923248   ||E||_F = 1.504817   ratio 0.6135
  trial 2: sqrt(sum of squared shifts) = 0.454650   ||E||_F = 1.268882   ratio 0.3583
  trial 3: sqrt(sum of squared shifts) = 0.687955   ||E||_F = 1.729274   ratio 0.3978


### Problem L3.3 — The Implicit Q theorem

**Statement.** Let $Q, V$ be orthogonal with $Q^{\top}AQ = H$ and $V^{\top}AV = G$ both
**unreduced** upper Hessenberg. Prove that $Qe_1 = Ve_1$ forces $V = QD$ and $G = DHD$ for some
$D = \operatorname{diag}(\pm 1)$.

**Intuition.** The Hessenberg relation is a recurrence that solves for column $j+1$ from column
$j$, and the recurrence has a non-zero pivot exactly when the form is unreduced.

**Solution.**

*Step 1 — a single orthogonal matrix.* Put $W = Q^{\top}V$. It is orthogonal, and
$We_1 = Q^{\top}Ve_1 = Q^{\top}Qe_1 = e_1$.

*Step 2 — an intertwining relation.*

$$
HW = Q^{\top}AQ\,Q^{\top}V = Q^{\top}AV = Q^{\top}V\,G = WG .
$$

*Step 3 — $W$ is upper triangular, by induction on its columns.* The claim is
$w_j \in \operatorname{span}(e_1, \dots, e_j)$. It holds for $j = 1$ by Step 1.

Assume it for $j$. Column $j$ of $HW = WG$ reads
$Hw_j = \sum_{i=1}^{j+1} g_{ij}w_i$, since $G$ is Hessenberg. Rearranged,

$$
g_{j+1,j}\, w_{j+1} = H w_j - \sum_{i=1}^{j} g_{ij} w_i .
$$

$H$ is Hessenberg and $w_j \in \operatorname{span}(e_1,\dots,e_j)$, so
$Hw_j \in \operatorname{span}(e_1,\dots,e_{j+1})$; the subtracted terms lie in
$\operatorname{span}(e_1,\dots,e_j)$. Because $G$ is unreduced, $g_{j+1,j} \neq 0$, so we may
divide and conclude $w_{j+1} \in \operatorname{span}(e_1,\dots,e_{j+1})$.

*Step 4 — orthogonal plus triangular equals signed identity.* Column $1$ of $W$ is a unit vector in
$\operatorname{span}(e_1)$, so $w_1 = \pm e_1$; inductively $w_j$ is a unit vector in
$\operatorname{span}(e_1,\dots,e_j)$ orthogonal to $w_1,\dots,w_{j-1} = \pm e_1,\dots,\pm e_{j-1}$,
hence $w_j = \pm e_j$. So $W = D$ with $D = \operatorname{diag}(\pm1)$.

*Step 5.* $V = QW = QD$, and $G = W^{\top}HW = DHD$.

$$
\boxed{Qe_1 = Ve_1 \ \Longrightarrow\ V = QD, \quad G = DHD, \quad D = \operatorname{diag}(\pm1)}
$$

**Key takeaway.** The whole reduction is determined by its first column. That is what licenses
bulge chasing: match the first column of the shifted step, restore the Hessenberg shape by
rotations, and the theorem guarantees you have performed the shifted QR step exactly — without
forming $A - \mu I$ or its QR factorization.

In [34]:
def hessenberg_from(A, q1):
    """Householder-free Arnoldi reduction to Hessenberg form starting from q1."""
    nn = A.shape[0]
    Qh = np.zeros((nn, nn))
    Hh = np.zeros((nn, nn))
    Qh[:, 0] = q1 / np.linalg.norm(q1)
    for j in range(nn):
        w = A @ Qh[:, j]
        for i in range(j + 1):
            Hh[i, j] = Qh[:, i] @ w
            w = w - Hh[i, j] * Qh[:, i]
        w = w - Qh[:, :j + 1] @ (Qh[:, :j + 1].T @ w)
        if j + 1 < nn:
            Hh[j + 1, j] = np.linalg.norm(w)
            Qh[:, j + 1] = w / Hh[j + 1, j]
    return Qh, Hh


Ai = rng.standard_normal((5, 5))
q1 = rng.standard_normal(5)
Q1h, H1h = hessenberg_from(Ai, q1)
Q2h, H2h = hessenberg_from(Ai, 2.5 * q1)          # same first column up to scale
D = np.diag(np.sign(np.diag(Q1h.T @ Q2h)))
print("subdiagonals of H1:", np.diag(H1h, -1))
print("W = Q1^T Q2 is diagonal with +-1 entries:\n", np.round(Q1h.T @ Q2h, 10))
print("||Q2 - Q1 D||_F =", np.linalg.norm(Q2h - Q1h @ D))
print("||H2 - D H1 D||_F =", np.linalg.norm(H2h - D @ H1h @ D))
assert np.allclose(Q1h.T @ Q2h, D, atol=1e-10)
assert np.linalg.norm(H2h - D @ H1h @ D) < 1e-10

subdiagonals of H1: [1.5083 3.2142 0.1577 0.2782]
W = Q1^T Q2 is diagonal with +-1 entries:
 [[ 1.  0.  0.  0. -0.]
 [-0.  1. -0.  0. -0.]
 [ 0.  0.  1.  0. -0.]
 [ 0.  0. -0.  1.  0.]
 [-0.  0.  0. -0.  1.]]


||Q2 - Q1 D||_F = 3.925844416168177e-16
||H2 - D H1 D||_F = 1.8531913762025007e-15


### Problem L3.4 — Linear convergence of inverse iteration, general diagonalizable case

**Statement.** Let $A$ be diagonalizable with eigenpairs $(\lambda_i, v_i)$, let
$\mu \notin \operatorname{spec}(A)$, and suppose the index $j$ uniquely minimizes
$\lvert \lambda_i - \mu \rvert$. Let $\lambda_\ell$ be the second closest. Prove that the inverse
iterates satisfy

$$
\operatorname{dist}(x_k, \operatorname{span}(v_j))
= O\!\left( \left\lvert \frac{\lambda_j - \mu}{\lambda_\ell - \mu} \right\rvert^{k} \right),
$$

provided the expansion of $x_0$ has a non-zero $v_j$ component.

**Intuition.** $(A - \mu I)^{-1}$ has the same eigenvectors as $A$ and eigenvalues
$(\lambda_i-\mu)^{-1}$; the largest of those belongs to the eigenvalue nearest $\mu$.

**Solution.**

*Step 1 — spectrum of the shifted inverse.* $Av_i = \lambda_i v_i$ gives
$(A - \mu I)^{-1}v_i = (\lambda_i - \mu)^{-1}v_i$.

*Step 2 — expand and apply.* With $x_0 = \sum_i c_i v_i$, $c_j \neq 0$,

$$
y_k = (A-\mu I)^{-k}x_0 = \sum_i \frac{c_i}{(\lambda_i - \mu)^{k}} v_i .
$$

*Step 3 — factor out the dominant term.*

$$
y_k = \frac{c_j}{(\lambda_j - \mu)^{k}}
\left[ v_j + \sum_{i \neq j} \frac{c_i}{c_j}\left( \frac{\lambda_j - \mu}{\lambda_i - \mu} \right)^{k} v_i \right] .
$$

*Step 4 — bound the tail.* Every $i \neq j$ has
$\lvert \lambda_i - \mu \rvert \ge \lvert \lambda_\ell - \mu \rvert$, so each coefficient is at
most $\lvert (\lambda_j-\mu)/(\lambda_\ell-\mu) \rvert^{k}$ in modulus, and the bracket is
$v_j + O(r^{k})$ with $r = \lvert (\lambda_j-\mu)/(\lambda_\ell-\mu) \rvert \lt 1$.

*Step 5 — normalize.* Normalization removes the scalar prefactor, so
$x_k = \bigl(v_j + O(r^{k})\bigr)/\lVert \cdot \rVert$ and the distance to
$\operatorname{span}(v_j)$ is $O(r^{k})$. In the non-symmetric case the constant carries
$\sum_{i \neq j}\lvert c_i/c_j \rvert \lVert v_i \rVert$, which is where the eigenvector condition
number enters.

$$
\boxed{r = \left\lvert \frac{\lambda_j - \mu}{\lambda_\ell - \mu} \right\rvert}
$$

**Key takeaway.** Theorem 4.1 is the symmetric case, where the constant is
$\tan\theta(x_0, q_j)$ and no conditioning factor appears. Here the rate is the same but the
constant is not, which is the general pattern for non-normal matrices.

In [35]:
lam = np.array([6.0, 2.2, 2.0, -4.0])
Xb = np.eye(4) + 0.4 * rng.standard_normal((4, 4))          # non-orthogonal eigenvectors
A = Xb @ np.diag(lam) @ np.linalg.inv(Xb)
mu = 2.05
dist = np.abs(lam - mu)
j = int(np.argmin(dist))
ell = int(np.argsort(dist)[1])
r = dist[j] / dist[ell]
vj = Xb[:, j] / np.linalg.norm(Xb[:, j])
x = rng.standard_normal(4)
x /= np.linalg.norm(x)
ds = []
for _ in range(25):
    ds.append(np.linalg.norm(x - (vj @ x) * vj))
    y = np.linalg.solve(A - mu * np.eye(4), x)
    x = y / np.linalg.norm(y)
obs = np.exp(np.polyfit(np.arange(6, 20), np.log(np.array(ds)[6:20]), 1)[0])
print(f"closest eigenvalue {lam[j]}, second closest {lam[ell]}")
print(f"observed rate {obs:.6f}   predicted {r:.6f}")
print(f"converged Rayleigh quotient {x @ A @ x:.8f}   (non-symmetric, so only an estimate)")
assert abs(obs - r) < 1e-3

closest eigenvalue 2.0, second closest 2.2


observed rate 0.333330   predicted 0.333333
converged Rayleigh quotient 2.00000000   (non-symmetric, so only an estimate)


### Problem L3.5 — Convergence of unshifted QR on a symmetric positive definite matrix

**Statement.** Let $A$ be symmetric positive definite with distinct eigenvalues
$\lambda_1 \gt \lambda_2 \gt \dots \gt \lambda_n \gt 0$ and spectral decomposition
$A = V\Lambda V^{\top}$. Assume additionally that $V^{\top}$ admits an LU factorization **without
pivoting**, that is every leading principal minor of $V^{\top}$ is non-zero. Prove that the
unshifted QR iterates satisfy

$$
\lvert (A_k)_{j+1,j} \rvert = O\!\left( \left( \frac{\lambda_{j+1}}{\lambda_j} \right)^{k} \right)
\quad \text{for every } j, \qquad (A_k)_{jj} \to \lambda_j .
$$

**Intuition.** QR iteration is subspace iteration on the coordinate flag; the $j$-th nested
subspace converges to the span of the top $j$ eigenvectors at the rate of the adjacent eigenvalue
ratio, and the entries below the diagonal measure how far that convergence still has to go.

**Solution.**

*Step 1 — the two identities.* By Problem L3.6, $A^{k} = \underline{Q}_k\underline{R}_k$ and
$A_k = \underline{Q}_k^{\top}A\underline{Q}_k$.

*Step 2 — factor the power.* $A^{k} = V\Lambda^{k}V^{\top} = V\Lambda^{k}LU$ with $L$ unit lower
triangular and $U$ upper triangular invertible.

*Step 3 — where the LU hypothesis is used.* Restricting to the first $j$ columns and using that
$U$ is upper triangular, $UE_j = E_jU_{11}$ with $U_{11}$ invertible, so

$$
\operatorname{Col}(A^{k}E_j) = \operatorname{Col}\bigl( V\Lambda^{k}LE_j \bigr) .
$$

Writing $LE_j = \left(\begin{smallmatrix}L_{11} \\ L_{21}\end{smallmatrix}\right)$, the block
$L_{11}$ is unit lower triangular hence **invertible** — this is precisely the pivot-free LU
hypothesis, and without it the argument collapses.

*Step 4 — the deviation matrix.* Multiplying on the right by $(\Lambda_1^{k}L_{11})^{-1}$,

$$
\operatorname{Col}(A^{k}E_j) = \operatorname{Col}\left( V \begin{pmatrix} I_j \\ G_k \end{pmatrix} \right),
\qquad G_k = \Lambda_2^{k}\,L_{21}L_{11}^{-1}\,\Lambda_1^{-k} .
$$

*Step 5 — the rate, stated without a free index.* Entrywise
$(G_k)_{pq} = N_{pq}(\lambda_{j+p}/\lambda_q)^{k}$ with $N = L_{21}L_{11}^{-1}$ fixed. Since
$\lambda_{j+p} \le \lambda_{j+1}$ and $\lambda_q \ge \lambda_j$ for $p \ge 1$, $q \le j$,

$$
\lVert G_k \rVert_F \;\le\; \lVert N \rVert_F \left( \frac{\lambda_{j+1}}{\lambda_j} \right)^{k} .
$$

The bound depends only on $j$, which fixes the free index $i$ of the naive statement
"$\lVert E_k \rVert = O((\lambda_{i+1}/\lambda_i)^{k})$": a norm of a single matrix cannot carry a
free index. The *per-entry* rate belongs to $(A_k)_{j+1,j}$, not to a matrix norm.

*Step 6 — from the subspace to the entries.* Since $V$ is orthogonal here, the columns of
$B_k = V\left(\begin{smallmatrix}I \\ G_k\end{smallmatrix}\right)$ have
$\sigma_{\min}(B_k) \ge 1$, so writing $\underline{Q}_k^{(j)} = B_kC_k$ gives
$\lVert C_k \rVert_2 \le 1$. Applying $A$,

$$
A\underline{Q}_k^{(j)} = \underline{Q}_k^{(j)}\bigl( C_k^{-1}\Lambda_1C_k \bigr)
+ V \begin{pmatrix} 0 \\ \Lambda_2G_k - G_k\Lambda_1 \end{pmatrix} C_k ,
$$

and multiplying on the left by the complementary columns $(\underline{Q}_k^{(j\perp)})^{\top}$
kills the first term, leaving

$$
\left\lVert (A_k)_{j+1:n,\,1:j} \right\rVert_2 \le 2\lambda_1 \lVert N \rVert_F
\left( \frac{\lambda_{j+1}}{\lambda_j} \right)^{k} .
$$

*Step 7 — the diagonal.* Multiplying on the left by $(\underline{Q}_k^{(j)})^{\top}$ instead and
taking traces gives $\sum_{i \le j}(A_k)_{ii} = \sum_{i \le j}\lambda_i + O((\lambda_{j+1}/\lambda_j)^{k})$;
subtracting the identity for $j-1$ isolates $(A_k)_{jj} \to \lambda_j$.

$$
\boxed{\lvert (A_k)_{j+1,j} \rvert = O\!\left( \left( \frac{\lambda_{j+1}}{\lambda_j} \right)^{k} \right),
\qquad (A_k)_{jj} \to \lambda_j}
$$

**Key takeaway.** Two corrections to the version of this result that circulates without
hypotheses: the pivot-free LU of $V^{\top}$ is required, and it selects the *ordering* of the
limit rather than the speed; and the decay rate is a property of one entry, so it cannot be
written as a norm bound carrying a free index. Section 7.3 of the theory notebook runs a
$2 \times 2$ where the hypothesis fails and the limit is ordered backwards.

In [36]:
spec = np.array([9.0, 4.5, 1.8, 0.6])
Vq, _ = np.linalg.qr(rng.standard_normal((4, 4)))
A = Vq @ np.diag(spec) @ Vq.T
A = (A + A.T) / 2.0
minors = [np.linalg.det(Vq.T[:i, :i]) for i in range(1, 5)]
print("leading principal minors of V^T:", np.round(minors, 6), " (all non-zero: pivot-free LU exists)")
K = 40
sub = np.zeros((K + 1, 3))
Ak = A.copy()
for k in range(K + 1):
    for i in range(3):
        sub[k, i] = abs(Ak[i + 1, i])
    Q, R = qr_positive(Ak)
    Ak = R @ Q
pred = spec[1:] / spec[:-1]
for i in range(3):
    obs = np.exp(np.polyfit(np.arange(12, 30), np.log(sub[12:30, i]), 1)[0])
    print(f"  subdiagonal ({i+2},{i+1}): observed {obs:.6f}   predicted {pred[i]:.6f}")
    assert abs(obs - pred[i]) < 1e-3
print("diagonal after 40 steps:", np.diag(Ak), "  true:", spec)
assert np.allclose(np.diag(Ak), spec, atol=1e-6)

leading principal minors of V^T: [-0.7576  0.3633 -0.4383 -1.    ]  (all non-zero: pivot-free LU exists)
  subdiagonal (2,1): observed 0.500000   predicted 0.500000
  subdiagonal (3,2): observed 0.400000   predicted 0.400000
  subdiagonal (4,3): observed 0.333333   predicted 0.333333
diagonal after 40 steps: [9.  4.5 1.8 0.6]   true: [9.  4.5 1.8 0.6]


### Problem L3.6 — The QR algorithm computes the QR factorization of $A^{k}$

**Statement.** With $A_0 = A$, $A_i = Q_iR_i$ and $A_{i+1} = R_iQ_i$, prove

$$
A^{k} = \underline{Q}_k \underline{R}_k, \qquad
\underline{Q}_k = Q_0Q_1\cdots Q_{k-1}, \quad \underline{R}_k = R_{k-1}\cdots R_1R_0 ,
$$

and $A_k = \underline{Q}_k^{\top}A\underline{Q}_k$.

**Intuition.** Each step is a similarity by $Q_k$; accumulating the similarities and the triangular
factors reassembles the power.

**Solution.**

*Step 1 — similarity.* $A_i = Q_iR_i$ and orthogonality give $R_i = Q_i^{\top}A_i$, hence

$$
A_{i+1} = R_iQ_i = Q_i^{\top}A_iQ_i .
$$

Chaining from $A_0 = A$ gives $A_k = \underline{Q}_k^{\top}A\underline{Q}_k$.

*Step 2 — base case.* $A^{1} = A = Q_0R_0 = \underline{Q}_1\underline{R}_1$.

*Step 3 — inductive step.* Assume $A^{k} = \underline{Q}_k\underline{R}_k$. Then

$$
\underline{Q}_{k+1}\underline{R}_{k+1} = \underline{Q}_kQ_k\,R_k\underline{R}_k
= \underline{Q}_k A_k \underline{R}_k ,
$$

because $Q_kR_k = A_k$ by definition.

*Step 4 — substitute the similarity.* Using $A_k = \underline{Q}_k^{\top}A\underline{Q}_k$ and
$\underline{Q}_k\underline{Q}_k^{\top} = I$,

$$
\underline{Q}_kA_k\underline{R}_k
= \underline{Q}_k\underline{Q}_k^{\top}A\underline{Q}_k\underline{R}_k
= A\,\underline{Q}_k\underline{R}_k = A\cdot A^{k} = A^{k+1} .
$$

$$
\boxed{A^{k} = \underline{Q}_k\underline{R}_k, \qquad A_k = \underline{Q}_k^{\top}A\underline{Q}_k}
$$

**Key takeaway.** The QR algorithm is orthonormal subspace iteration on all $n$ nested coordinate
subspaces at once, but it never forms $A^{k}$, whose entries grow like $\rho(A)^{k}$ and overflow.
Everything the algorithm touches has the size of $A$.

In [37]:
Ar = rng.standard_normal((4, 4)) + 2.5 * np.eye(4)
Ak = Ar.copy()
Qacc = np.eye(4)
Racc = np.eye(4)
for k in range(1, 11):
    Q, R = qr_positive(Ak)
    Ak = R @ Q
    Qacc = Qacc @ Q
    Racc = R @ Racc
    if k in (1, 5, 10):
        Apow = np.linalg.matrix_power(Ar, k)
        e1 = np.linalg.norm(Apow - Qacc @ Racc) / np.linalg.norm(Apow)
        e2 = np.linalg.norm(Ak - Qacc.T @ Ar @ Qacc) / np.linalg.norm(Ar)
        print(f"  k = {k:2d}:  ||A^k - Q R||/||A^k|| = {e1:.2e} = {e1/EPS:.1f} eps"
              f"   ||A_k - Q^T A Q||/||A|| = {e2:.2e}")
        assert e1 < 1e-12 and e2 < 1e-12

  k =  1:  ||A^k - Q R||/||A^k|| = 2.31e-16 = 1.0 eps   ||A_k - Q^T A Q||/||A|| = 1.44e-16
  k =  5:  ||A^k - Q R||/||A^k|| = 1.44e-15 = 6.5 eps   ||A_k - Q^T A Q||/||A|| = 2.32e-16
  k = 10:  ||A^k - Q R||/||A^k|| = 4.26e-15 = 19.2 eps   ||A_k - Q^T A Q||/||A|| = 5.26e-16


### Problem L3.7 — Why a shift accelerates the trailing subdiagonal

**Statement.** Explain, with the rate, why choosing $\mu_k \approx \lambda_n$ in the shifted QR
step makes $(A_k)_{n,n-1}$ decay far faster than in the unshifted algorithm.

**Intuition.** Shifting translates the whole spectrum. The decay of the last subdiagonal entry is
governed by the ratio of the two smallest shifted moduli, and a good shift makes the numerator
tiny while leaving the denominator alone.

**Solution.**

*Step 1 — unshifted rate.* By Problem L3.5 with $j = n-1$,

$$
\lvert (A_k)_{n,n-1} \rvert = O\!\left( \left\lvert \frac{\lambda_n}{\lambda_{n-1}} \right\rvert^{k} \right),
$$

which is slow when the two smallest eigenvalues are close in modulus.

*Step 2 — the shifted step is a similarity.* From $A_k - \mu_kI = Q_kR_k$ and
$A_{k+1} = R_kQ_k + \mu_kI$,

$$
A_{k+1} = Q_k^{\top}(A_k - \mu_kI)Q_k + \mu_kI = Q_k^{\top}A_kQ_k ,
$$

so the spectrum is unchanged and only the shape moves.

*Step 3 — the rate in the shifted spectrum.* One step of the algorithm run on $A_k - \mu_kI$,
whose eigenvalues are $\lambda_i - \mu_k$, contracts the last subdiagonal entry by

$$
\left\lvert \frac{\lambda_n - \mu_k}{\lambda_{n-1} - \mu_k} \right\rvert .
$$

*Step 4 — a good shift.* If $\mu_k \to \lambda_n$ then the numerator tends to $0$ while the
denominator tends to $\lvert \lambda_{n-1} - \lambda_n \rvert \gt 0$, so the factor tends to $0$
and convergence stops being linear.

*Step 5 — how good.* With the Rayleigh quotient shift $\mu_k = (A_k)_{nn}$ the shift error is
itself quadratic in the residual, giving quadratic convergence; with the Wilkinson shift of
Definition 3.5 the symmetric tridiagonal case is cubic (Theorem 4.9).

$$
\boxed{\text{contraction factor} = \left\lvert \frac{\lambda_n - \mu_k}{\lambda_{n-1} - \mu_k} \right\rvert \longrightarrow 0}
$$

**Key takeaway.** A shift is an eigenvalue estimate fed back into the algorithm, and the feedback
is what turns a linear method into a cubic one. The price is that the shift must be chosen so it
cannot stagnate — which is exactly what the Wilkinson choice guarantees, and the naive corner-entry
choice does not on $\left(\begin{smallmatrix}0 & 1 \\ 1 & 0\end{smallmatrix}\right)$.

In [38]:
spec = np.array([5.0, 3.0, 2.6, 2.4])
Vs, _ = np.linalg.qr(rng.standard_normal((4, 4)))
A = Vs @ np.diag(spec) @ Vs.T
A = (A + A.T) / 2.0
print("unshifted, predicted rate |lambda_4/lambda_3| =", spec[3] / spec[2])
Ak = A.copy()
for k in range(1, 21):
    Q, R = qr_positive(Ak)
    Ak = R @ Q
    if k in (5, 10, 20):
        print(f"  unshifted step {k:2d}: |(A_k)_(4,3)| = {abs(Ak[3,2]):.3e}")
un = abs(Ak[3, 2])
Ak = A.copy()
for k in range(1, 7):
    mu = wilkinson_shift(Ak[2, 2], Ak[3, 2], Ak[3, 3])
    Q, R = qr_positive(Ak - mu * np.eye(4))
    Ak = R @ Q + mu * np.eye(4)
    print(f"  Wilkinson-shifted step {k}: mu = {mu:.10f}   |(A_k)_(4,3)| = {abs(Ak[3,2]):.3e}")
assert abs(Ak[3, 2]) < 1e-14 < un

unshifted, predicted rate |lambda_4/lambda_3| = 0.923076923076923
  unshifted step  5: |(A_k)_(4,3)| = 9.793e-02
  unshifted step 10: |(A_k)_(4,3)| = 9.281e-02
  unshifted step 20: |(A_k)_(4,3)| = 5.555e-02
  Wilkinson-shifted step 1: mu = 2.6037338998   |(A_k)_(4,3)| = 1.410e-03
  Wilkinson-shifted step 2: mu = 2.5999727022   |(A_k)_(4,3)| = 3.243e-07
  Wilkinson-shifted step 3: mu = 2.6000000000   |(A_k)_(4,3)| = 2.083e-17
  Wilkinson-shifted step 4: mu = 2.6000000000   |(A_k)_(4,3)| = 1.088e-48
  Wilkinson-shifted step 5: mu = 2.6000000000   |(A_k)_(4,3)| = 4.095e-111
  Wilkinson-shifted step 6: mu = 2.6000000000   |(A_k)_(4,3)| = 5.400e-236


### Problem L3.8 — The Lanczos three-term recurrence

**Statement.** Derive $\beta_j q_{j+1} = Aq_j - \alpha_jq_j - \beta_{j-1}q_{j-1}$ for symmetric
$A$, starting from the Arnoldi relation $AQ_m = Q_mH_m + h_{m+1,m}q_{m+1}e_m^{\top}$.

**Intuition.** Symmetry makes the projected matrix symmetric, and a symmetric Hessenberg matrix has
nothing outside three diagonals.

**Solution.**

*Step 1 — the projected matrix.* Multiplying the Arnoldi relation by $Q_m^{\top}$ and using
$Q_m^{\top}Q_m = I$, $Q_m^{\top}q_{m+1} = 0$, gives $H_m = Q_m^{\top}AQ_m$.

*Step 2 — it is symmetric.*

$$
H_m^{\top} = (Q_m^{\top}AQ_m)^{\top} = Q_m^{\top}A^{\top}Q_m = Q_m^{\top}AQ_m = H_m .
$$

*Step 3 — Hessenberg plus symmetric equals tridiagonal.* $h_{ij} = 0$ for $i \gt j+1$ by the
Hessenberg property, and $h_{ij} = h_{ji}$ then forces $h_{ij} = 0$ for $j \gt i+1$ as well. So
$h_{ij} = 0$ whenever $\lvert i-j \rvert \gt 1$ and $H_m = T_m$ is tridiagonal.

*Step 4 — read column $j$.* Write $\alpha_j = h_{jj}$ and $\beta_j = h_{j+1,j}$; symmetry gives
$h_{j-1,j} = h_{j,j-1} = \beta_{j-1}$. Column $j$ of
$AQ_m = Q_mT_m + \beta_mq_{m+1}e_m^{\top}$ reads

$$
Aq_j = \beta_{j-1}q_{j-1} + \alpha_jq_j + \beta_jq_{j+1} .
$$

*Step 5 — rearrange.*

$$
\beta_jq_{j+1} = Aq_j - \alpha_jq_j - \beta_{j-1}q_{j-1} .
$$

$$
\boxed{\beta_jq_{j+1} = Aq_j - \alpha_jq_j - \beta_{j-1}q_{j-1},
\quad \alpha_j = q_j^{\top}Aq_j, \quad \beta_j = \lVert \text{right side} \rVert_2}
$$

**Key takeaway.** Arnoldi orthogonalizes against $j$ vectors at step $j$, costing $O(m^{2}n)$ and
$O(mn)$ memory; Lanczos orthogonalizes against two, costing $O(mn)$ and $O(n)$ memory. In exact
arithmetic they produce the same basis. In floating point they do not, and the difference is
Theorem 4.9.

In [39]:
Mt = rng.standard_normal((9, 9))
As = (Mt + Mt.T) / 2.0
b = rng.standard_normal(9)
m = 5
a_t, b_t, Qt = lanczos(As, b, m, reorth=True)
Tm = np.diag(a_t) + np.diag(b_t[:-1], 1) + np.diag(b_t[:-1], -1)
print("T_m =\n", Tm)
print("H_m = Q^T A Q equals T_m to",
      np.abs(Qt[:, :m].T @ As @ Qt[:, :m] - Tm).max(), "absolute error")
res = As @ Qt[:, :m] - Qt[:, :m] @ Tm - b_t[m - 1] * np.outer(Qt[:, m], np.eye(m)[m - 1])
print("||A Q_m - Q_m T_m - beta_m q_(m+1) e_m^T||_F =", np.linalg.norm(res))
for j in range(1, m):
    lhs = b_t[j] * Qt[:, j + 1]
    rhs = As @ Qt[:, j] - a_t[j] * Qt[:, j] - b_t[j - 1] * Qt[:, j - 1]
    assert np.linalg.norm(lhs - rhs) < 1e-11
print("three-term recurrence verified for every j")
assert np.linalg.norm(res) < 1e-11

T_m =
 [[-0.1558  1.8243  0.      0.      0.    ]
 [ 1.8243  0.9633  2.2644  0.      0.    ]
 [ 0.      2.2644  0.2227  2.4897  0.    ]
 [ 0.      0.      2.4897  1.2594  1.5088]
 [ 0.      0.      0.      1.5088  0.9752]]
H_m = Q^T A Q equals T_m to 4.440892098500626e-16 absolute error
||A Q_m - Q_m T_m - beta_m q_(m+1) e_m^T||_F = 9.738769112036936e-16
three-term recurrence verified for every j


### Problem L3.9 — Ritz values are the eigenvalues of the compressed operator

**Statement.** Let $Q_m$ be an orthonormal basis of $\mathcal{K}_m(A,b)$ and
$\Pi = Q_mQ_m^{\top}$ the orthogonal projector onto it. Prove that the Ritz values are exactly the
non-trivial eigenvalues of $\Pi A \Pi$ restricted to $\mathcal{K}_m$, that is the eigenvalues of
$H_m = Q_m^{\top}AQ_m$.

**Intuition.** Restricting an operator to a subspace and then projecting back is what
"compressing" means, and in the basis $Q_m$ the compressed operator is the small matrix $H_m$.

**Solution.**

*Step 1 — parameterize.* Every $y \in \mathcal{K}_m$ is $y = Q_mz$ for a unique
$z \in \mathbb{R}^{m}$, and $y \neq 0$ iff $z \neq 0$.

*Step 2 — write the eigenvalue equation.* $\Pi A\Pi y = \theta y$ with $y = Q_mz$ becomes

$$
Q_mQ_m^{\top}A\,Q_mQ_m^{\top}Q_mz = \theta Q_mz .
$$

*Step 3 — simplify.* $Q_m^{\top}Q_m = I_m$, so the left side is $Q_m(Q_m^{\top}AQ_m)z = Q_mH_mz$.

*Step 4 — strip $Q_m$.* Multiplying on the left by $Q_m^{\top}$ and using $Q_m^{\top}Q_m = I_m$
again gives $H_mz = \theta z$; conversely $H_mz = \theta z$ implies
$\Pi A\Pi(Q_mz) = \theta(Q_mz)$.

*Step 5 — Galerkin reading.* The condition $H_mz = \theta z$ is equivalent to
$Q_m^{\top}(Ay - \theta y) = 0$, that is $Ay - \theta y \perp \mathcal{K}_m$. Ritz pairs are
exactly the pairs whose residual is orthogonal to the search space.

$$
\boxed{\theta \text{ is a Ritz value} \iff H_mz = \theta z \iff Ay - \theta y \perp \mathcal{K}_m}
$$

**Key takeaway.** The Rayleigh-Ritz procedure is a Galerkin method. Theorem 4.8 then says the
residual, which lies entirely along $q_{m+1}$, has the explicitly known length
$\beta_m\lvert s_{mi} \rvert$ — and Problem L3.1 says the Ritz values are trapped inside the
spectrum.

In [40]:
Mg = rng.standard_normal((10, 10))
Ag = (Mg + Mg.T) / 2.0
bg = rng.standard_normal(10)
m = 4
a_g, b_g, Qg = lanczos(Ag, bg, m, reorth=True)
Qm = Qg[:, :m]
Hm = Qm.T @ Ag @ Qm
Pi = Qm @ Qm.T
th_small = np.linalg.eigvalsh(Hm)
th_proj = np.linalg.eigvalsh(Pi @ Ag @ Pi)
print("eigenvalues of H_m           :", np.round(th_small, 8))
print("non-zero eigenvalues of Pi A Pi:", np.round(th_proj[np.abs(th_proj) > 1e-10], 8))
for i in range(m):
    z = np.linalg.eigh(Hm)[1][:, i]
    y = Qm @ z
    print(f"  Ritz pair {i}: ||Q_m^T (A y - theta y)|| = {np.linalg.norm(Qm.T @ (Ag @ y - th_small[i] * y)):.2e}")
    assert np.linalg.norm(Qm.T @ (Ag @ y - th_small[i] * y)) < 1e-10
assert np.allclose(np.sort(th_small), np.sort(th_proj[np.abs(th_proj) > 1e-10]))

eigenvalues of H_m           : [-1.4522  0.267   1.6244  4.6502]
non-zero eigenvalues of Pi A Pi: [-1.4522  0.267   1.6244  4.6502]
  Ritz pair 0: ||Q_m^T (A y - theta y)|| = 1.22e-15
  Ritz pair 1: ||Q_m^T (A y - theta y)|| = 7.84e-16
  Ritz pair 2: ||Q_m^T (A y - theta y)|| = 9.21e-16
  Ritz pair 3: ||Q_m^T (A y - theta y)|| = 1.72e-15


### Problem L3.10 — Gershgorin's second theorem

**Statement.** If a union of $k$ Gershgorin discs of $A \in \mathbb{C}^{n \times n}$ is disjoint
from the other $n-k$ discs, prove that the union contains exactly $k$ eigenvalues, counted with
algebraic multiplicity.

**Intuition.** Deform the matrix continuously to its diagonal. Eigenvalues move continuously and
stay inside their discs, and they cannot cross the gap between two disjoint components.

**Solution.**

*Step 1 — a homotopy.* Write $A = D + B$ with $D = \operatorname{diag}(a_{11},\dots,a_{nn})$, and
set $A(t) = D + tB$ for $t \in [0,1]$, so $A(0) = D$ and $A(1) = A$.

*Step 2 — shrinking discs.* The Gershgorin discs of $A(t)$ have the same centres $a_{ii}$ and radii
$tR_i \le R_i$, so each disc of $A(t)$ is contained in the corresponding disc of $A$. If the union
$U$ of $k$ discs of $A$ is disjoint from the rest, the same holds for $A(t)$ at every $t$.

*Step 3 — continuity.* The coefficients of $\det(A(t) - zI)$ are polynomials in $t$, and the roots
of a monic polynomial depend continuously on its coefficients, so the $n$ eigenvalues of $A(t)$ can
be chosen as continuous functions $\lambda_1(t), \dots, \lambda_n(t)$.

*Step 4 — count at $t = 0$.* $A(0) = D$ has eigenvalues $a_{11},\dots,a_{nn}$, exactly $k$ of which
are the centres of the discs forming $U$, hence lie in $U$; the other $n-k$ lie in the complementary
component.

*Step 5 — no crossing.* Each $\lambda_i(t)$ lies in the union of all discs of $A(t)$, which is
contained in $U \cup U^{c}$ with $U$ and $U^{c}$ disjoint closed sets. A continuous path in a
disjoint union of two closed sets stays in the component it starts in.

*Step 6 — conclude.* At $t = 1$ exactly $k$ eigenvalues lie in $U$.

$$
\boxed{\text{a union of } k \text{ discs disjoint from the rest contains exactly } k \text{ eigenvalues}}
$$

**Key takeaway.** Containment is cheap; **counting** is what makes Gershgorin usable as a shift
strategy, because it tells you how many eigenvalues a deflation will produce before you compute
any of them.

In [41]:
G = np.array([[10.0, 0.4, 0.2, 0.0],
              [0.3, 9.5, 0.1, 0.2],
              [0.0, 0.1, 1.0, 0.5],
              [0.1, 0.0, 0.4, 0.5]])
centres = np.diag(G)
radii = np.abs(G).sum(axis=1) - np.abs(centres)
ev = np.linalg.eigvals(G)
print("centres:", centres, " radii:", radii)
print("eigenvalues:", np.round(ev, 6))
group = [0, 1]
inside = [z for z in ev if min(abs(z - centres[i]) - radii[i] for i in group) <= 1e-12]
print(f"discs {[g+1 for g in group]} form a component disjoint from the rest;"
      f" it contains {len(inside)} eigenvalues")
for t in (0.0, 0.25, 0.5, 0.75, 1.0):
    At = np.diag(centres) + t * (G - np.diag(centres))
    cnt = sum(1 for z in np.linalg.eigvals(At)
              if min(abs(z - centres[i]) - t * radii[i] for i in group) <= 1e-9)
    print(f"  t = {t:.2f}: eigenvalues inside the first component = {cnt}")
    assert cnt == 2
assert len(inside) == 2

centres: [10.   9.5  1.   0.5]  radii: [0.6 0.6 0.6 0.5]
eigenvalues: [10.1793  9.3219  0.2393  1.2595]
discs [1, 2] form a component disjoint from the rest; it contains 2 eigenvalues
  t = 0.00: eigenvalues inside the first component = 2
  t = 0.25: eigenvalues inside the first component = 2
  t = 0.50: eigenvalues inside the first component = 2
  t = 0.75: eigenvalues inside the first component = 2
  t = 1.00: eigenvalues inside the first component = 2


### Problem L3.11 — The bilinear characterization of the largest singular value

**Statement.** For $A \in \mathbb{R}^{m \times n}$ prove

$$
\max_{u \neq 0,\ v \neq 0} \frac{u^{\top}Av}{\lVert u \rVert_2 \lVert v \rVert_2} = \sigma_1(A) .
$$

**Intuition.** Fix $v$ and the best $u$ is the direction of $Av$; then the problem collapses to the
operator norm.

**Solution.**

*Step 1 — upper bound in $u$.* Cauchy-Schwarz gives $u^{\top}(Av) \le \lVert u \rVert \lVert Av \rVert$
with equality iff $u \parallel Av$, so for fixed $v$,

$$
\max_{u \neq 0} \frac{u^{\top}Av}{\lVert u \rVert \lVert v \rVert} = \frac{\lVert Av \rVert}{\lVert v \rVert} .
$$

*Step 2 — maximize in $v$.* By definition of the operator norm,
$\max_{v \neq 0} \lVert Av \rVert / \lVert v \rVert = \lVert A \rVert_{\mathrm{op}}$, and the SVD
gives $\lVert A \rVert_{\mathrm{op}} = \sigma_1(A)$.

*Step 3 — attainment.* With $A = U\Sigma V^{\top}$, take $v = v_1$ and $u = u_1$: then
$Av_1 = \sigma_1u_1$ and $u_1^{\top}Av_1 = \sigma_1$.

$$
\boxed{\max_{u,v \neq 0} \frac{u^{\top}Av}{\lVert u \rVert_2\lVert v \rVert_2} = \sigma_1(A),
\quad \text{attained at } (u_1, v_1)}
$$

**Key takeaway.** This is the SVD analogue of the Rayleigh quotient, and alternating the two
maximizations — normalize $u \gets Av$, then $v \gets A^{\top}u$ — is exactly the power iteration
of Problem L2.10.

In [42]:
Ab = rng.standard_normal((7, 5))
sv = np.linalg.svd(Ab, compute_uv=False)
best = 0.0
for _ in range(4000):
    u = rng.standard_normal(7)
    v = rng.standard_normal(5)
    best = max(best, u @ Ab @ v / (np.linalg.norm(u) * np.linalg.norm(v)))
U, S, Vt = np.linalg.svd(Ab)
attained = U[:, 0] @ Ab @ Vt[0]
print(f"random search over 4000 pairs : {best:.6f}")
print(f"value at (u_1, v_1)           : {attained:.10f}")
print(f"sigma_1(A)                    : {sv[0]:.10f}")
assert abs(attained - sv[0]) < 1e-12 and best <= sv[0] + 1e-12

random search over 4000 pairs : 3.541233
value at (u_1, v_1)           : 4.5578207459
sigma_1(A)                    : 4.5578207459


### Problem L3.12 — Two-sided Rayleigh quotient iteration

**Statement.** For a non-symmetric $A$ with a simple eigenvalue $\lambda$, right eigenvector $v$
and left eigenvector $w$ normalized by $w^{\ast}v = 1$, show that the two-sided Rayleigh quotient
$R(x,y) = \dfrac{y^{\ast}Ax}{y^{\ast}x}$ has error $O(\epsilon_x\epsilon_y)$ when
$x = v + \epsilon_xe_x$, $y = w + \epsilon_ye_y$ with $e_x \perp w$, $e_y \perp v$, and deduce that
the resulting iteration is cubic.

**Intuition.** The one-sided quotient loses its quadratic accuracy for non-symmetric matrices
because left and right eigenvectors differ. Using both restores the cancellation.

**Solution.**

*Step 1 — numerator.*

$$
y^{\ast}Ax = (w + \epsilon_ye_y)^{\ast}A(v + \epsilon_xe_x)
= \lambda + \epsilon_x \lambda\, w^{\ast}e_x + \epsilon_y \lambda\, e_y^{\ast}v + \epsilon_x\epsilon_y\, e_y^{\ast}Ae_x ,
$$

using $Av = \lambda v$ for the third term and $w^{\ast}A = \lambda w^{\ast}$ for the second.

*Step 2 — the linear terms vanish.* $w^{\ast}e_x = 0$ and $e_y^{\ast}v = 0$ by hypothesis, so

$$
y^{\ast}Ax = \lambda + \epsilon_x\epsilon_y\, e_y^{\ast}Ae_x .
$$

*Step 3 — denominator.* The same two orthogonalities give
$y^{\ast}x = 1 + \epsilon_x\epsilon_y\, e_y^{\ast}e_x$.

*Step 4 — divide.*

$$
R(x,y) = \lambda + \epsilon_x\epsilon_y\bigl( e_y^{\ast}Ae_x - \lambda\, e_y^{\ast}e_x \bigr) + O\bigl( (\epsilon_x\epsilon_y)^{2} \bigr) .
$$

*Step 5 — the iteration.* Using $\mu = R(x,y)$ as the shift in *two* inverse iterations, one with
$A - \mu I$ for $x$ and one with $(A - \mu I)^{\ast}$ for $y$, Theorem 4.1 contracts each error by
a factor proportional to $\lvert \lambda - \mu \rvert = O(\epsilon_x\epsilon_y)$:

$$
\epsilon_x^{+} = O(\epsilon_x^{2}\epsilon_y), \qquad \epsilon_y^{+} = O(\epsilon_x\epsilon_y^{2}) .
$$

*Step 6 — order.* With $\epsilon = \max(\epsilon_x,\epsilon_y)$ this reads
$\epsilon^{+} = O(\epsilon^{3})$.

$$
\boxed{\mu - \lambda = O(\epsilon_x\epsilon_y), \qquad \epsilon^{+} = O(\epsilon^{3})}
$$

**Key takeaway.** Symmetry is not needed for cubic convergence; **biorthogonality** is. For
symmetric $A$ the left and right eigenvectors coincide and the two-sided iteration degenerates to
ordinary RQI (Theorem 4.2). The cost is one extra linear solve with $A^{\ast}$ per step.

In [43]:
lam_t = np.array([3.0, 1.0, -2.0, 0.5])
Xt = np.eye(4) + 0.5 * rng.standard_normal((4, 4))
A = Xt @ np.diag(lam_t) @ np.linalg.inv(Xt)
x = Xt[:, 0] + 0.4 * rng.standard_normal(4)
y = np.linalg.inv(Xt).T[:, 0] + 0.4 * rng.standard_normal(4)
x /= np.linalg.norm(x)
y /= np.linalg.norm(y)
errs = []
for k in range(6):
    mu = (y @ A @ x) / (y @ x)
    errs.append(abs(mu - 3.0))
    xn = np.linalg.solve(A - mu * np.eye(4), x)
    yn = np.linalg.solve((A - mu * np.eye(4)).T, y)
    x, y = xn / np.linalg.norm(xn), yn / np.linalg.norm(yn)
errs = np.array(errs)
print("two-sided RQI, |mu_k - lambda|:", errs)
usable = [k for k in range(len(errs) - 1) if 0 < errs[k] < 0.3 and errs[k + 1] > 1e-15]
for k in usable:
    print(f"  measured order at k = {k}: {np.log(errs[k+1])/np.log(errs[k]):.3f}")
assert errs[-1] < 1e-12

two-sided RQI, |mu_k - lambda|:

 [1.5962 0.2037 0.0016 0.     0.     0.    ]
  measured order at k = 1: 4.051
  measured order at k = 2: 3.218
  measured order at k = 3: 1.638
  measured order at k = 4: 1.000


### Problem L3.13 — Paige's theorem, and the residual identity behind it

**Statement.** (a) Prove that in exact arithmetic the Lanczos Ritz residual is
$\lVert Ay_i - \theta_iy_i \rVert_2 = \beta_m\lvert s_{mi} \rvert$ exactly. (b) State Paige's
finite-precision result relating $\lvert q_{m+1}^{\top}y_i \rvert$ to that residual, and explain
selective reorthogonalization.

**Intuition.** The whole Lanczos residual points along the single new direction $q_{m+1}$, so its
length is one number times one component.

**Solution.**

*Step 1 — the factorization.* Theorem 4.7 gives
$AQ_m = Q_mT_m + \beta_mq_{m+1}e_m^{\top}$.

*Step 2 — apply to $s_i$.* With $T_ms_i = \theta_is_i$, $\lVert s_i \rVert_2 = 1$ and
$y_i = Q_ms_i$,

$$
Ay_i = AQ_ms_i = Q_mT_ms_i + \beta_mq_{m+1}(e_m^{\top}s_i) = \theta_iy_i + \beta_ms_{mi}\,q_{m+1} .
$$

*Step 3 — take norms.* $\lVert q_{m+1} \rVert_2 = 1$, so

$$
\lVert Ay_i - \theta_iy_i \rVert_2 = \beta_m\lvert s_{mi} \rvert .
$$

This is an identity, computed from two numbers already in hand.

*Step 4 — Paige's result.* In floating point with unit roundoff $u$, the computed Lanczos vectors
satisfy

$$
\lvert q_{m+1}^{\top}y_i \rvert = O\!\left( \frac{u\,\lVert A \rVert_2}{\beta_m\lvert s_{mi} \rvert} \right) .
$$

The new basis vector loses orthogonality **in the direction of a converged Ritz vector**, and by
exactly the amount that its residual has shrunk. Orthogonality against Ritz vectors that have not
converged is preserved.

*Step 5 — selective reorthogonalization.* Rather than orthogonalizing against all $m$ previous
vectors at cost $O(m^{2}n)$, monitor $\beta_m\lvert s_{mi} \rvert$ — free, by part (a) — and
orthogonalize against $y_i$ only when

$$
\beta_m\lvert s_{mi} \rvert \le \sqrt{u}\,\lVert A \rVert_2 ,
$$

the threshold at which Step 4 predicts the loss reaches $\sqrt{u}$. This keeps the basis
semi-orthogonal, which is enough for accurate Ritz values, at a small fraction of the cost.

$$
\boxed{\lVert Ay_i - \theta_iy_i \rVert_2 = \beta_m\lvert s_{mi} \rvert;
\qquad \lvert q_{m+1}^{\top}y_i \rvert = O\!\left( \frac{u\lVert A \rVert_2}{\beta_m\lvert s_{mi} \rvert} \right)}
$$

**Key takeaway.** Part (a) is free and exact and gives a stopping test; part (b) says the same
number predicts when the algorithm is about to break. One quantity, two uses — which is why every
serious Lanczos implementation computes it every step.

In [44]:
nP = 100
specP = np.concatenate([[7.0, 5.5], np.linspace(0.2, 3.0, nP - 2)])
QP, _ = np.linalg.qr(rng.standard_normal((nP, nP)))
AP = QP @ np.diag(specP) @ QP.T
AP = (AP + AP.T) / 2.0
normA = np.linalg.norm(AP, 2)
bP = rng.standard_normal(nP)

print("(a) exact residual identity, with reorthogonalization")
mP = 12
a_p, b_p, Qp = lanczos(AP, bP, mP, reorth=True)
Tp = np.diag(a_p) + np.diag(b_p[:-1], 1) + np.diag(b_p[:-1], -1)
th, Sp = np.linalg.eigh(Tp)
for i in (0, mP // 2, mP - 1):
    yi = Qp[:, :mP] @ Sp[:, i]
    lhs = np.linalg.norm(AP @ yi - th[i] * yi)
    rhs = b_p[mP - 1] * abs(Sp[mP - 1, i])
    print(f"  theta = {th[i]:+.6f}   ||A y - theta y|| = {lhs:.12f}   beta_m |s_mi| = {rhs:.12f}")
    assert abs(lhs - rhs) < 1e-11

print("\n(b) finite precision, without reorthogonalization")
mQ = 40
a_q, b_q, Qq = lanczos(AP, bP, mQ, reorth=False)
Tq = np.diag(a_q) + np.diag(b_q[:-1], 1) + np.diag(b_q[:-1], -1)
thq, Sq = np.linalg.eigh(Tq)
i_top = int(np.argmax(thq))
y_top = Qq[:, :mQ] @ Sq[:, i_top]
resid = b_q[mQ - 1] * abs(Sq[mQ - 1, i_top])
overlap = abs(Qq[:, mQ] @ y_top)
print(f"  converged Ritz value {thq[i_top]:.12f}   (true 7.0)")
print(f"  residual beta_m |s_mi|          = {resid:.3e}")
print(f"  measured |q_(m+1)^T y_i|        = {overlap:.3e}")
print(f"  Paige estimate u ||A|| / residual = {min((EPS/2)*normA/max(resid,1e-300), 1.0):.3e}")
print(f"  selective threshold sqrt(u) ||A|| = {np.sqrt(EPS/2)*normA:.3e}")
assert overlap > 1e-6
assert resid < np.sqrt(EPS / 2) * normA

(a) exact residual identity, with reorthogonalization
  theta = +0.227386   ||A y - theta y|| = 0.034044573135   beta_m |s_mi| = 0.034044573135
  theta = +2.075247   ||A y - theta y|| = 0.311431165136   beta_m |s_mi| = 0.311431165136
  theta = +7.000000   ||A y - theta y|| = 0.000000765504   beta_m |s_mi| = 0.000000765504

(b) finite precision, without reorthogonalization
  converged Ritz value 7.000000000000   (true 7.0)
  residual beta_m |s_mi|          = 9.293e-14
  measured |q_(m+1)^T y_i|        = 1.551e-02
  Paige estimate u ||A|| / residual = 8.363e-03
  selective threshold sqrt(u) ||A|| = 7.376e-08
